# OOD Follow-up Optimization Experiments

这个 notebook 用来完成 OOD 实验后的后续优化：阈值校准、generator-level 错误分析、R5/R7 概率融合，以及少量重点重训实验。

运行建议：不要直接 Run All。先完成不重训步骤，再按需要逐个打开长训练 cell。长训练默认由 `RUN_LONG_TRAINING = False` 保护。

## 1. 环境与路径设置

本 notebook 仍然只做编排：训练用 `src.train`，评估用 `src.evaluate`，分析用 `scripts/ood_*.py`。

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
from datetime import datetime

import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
os.chdir(PROJECT_ROOT)

PYTHON = PROJECT_ROOT / ".venv" / "Scripts" / "python.exe"
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

OUT_DIR = PROJECT_ROOT / "outputs" / "ood_followup"
LOG_DIR = PROJECT_ROOT / "logs" / "ood_followup"
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

RUN_LONG_TRAINING = True   # 长训练保护开关：确认要跑某个重训实验时再改 True
FORCE_EVAL = False          # True 会覆盖已有 val/test predictions

print("PROJECT_ROOT =", PROJECT_ROOT)
print("PYTHON       =", PYTHON)
print("OUT_DIR      =", OUT_DIR)
print("RUN_LONG_TRAINING =", RUN_LONG_TRAINING)

PROJECT_ROOT = D:\VsCode Program\Python\content_security\final_project
PYTHON       = D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe
OUT_DIR      = D:\VsCode Program\Python\content_security\final_project\outputs\ood_followup
RUN_LONG_TRAINING = True


## 2. 基础实验配置

这些目录对应已经完成的 OOD R1-R7 基线。后续脚本默认读取这里的 `best.pt` 和 test `predictions.csv`。

In [2]:
TRAIN_CSV = "data/GenVideo-Val/splits/ood_train.csv"
VAL_CSV = "data/GenVideo-Val/splits/ood_val.csv"
TEST_CSV = "data/GenVideo-Val/splits/ood_test.csv"

BASE_EXPERIMENTS = {
    "R1": {"method": "frequency_wave", "model": "frequency_wave", "run_dir": "runs/ood_frequency_wave", "out_dir": "outputs/ood_frequency_wave"},
    "R2": {"method": "spatial_dino", "model": "spatial_dino", "run_dir": "runs/ood_spatial_dino", "out_dir": "outputs/ood_spatial_dino"},
    "R3": {"method": "temporal_d3_restrav", "model": "temporal_d3_restrav", "run_dir": "runs/ood_temporal_d3_restrav", "out_dir": "outputs/ood_temporal_d3_restrav"},
    "R4": {"method": "spatial_frequency_new", "model": "spatial_frequency_new", "run_dir": "runs/ood_spatial_frequency_new", "out_dir": "outputs/ood_spatial_frequency_new"},
    "R5": {"method": "spatial_temporal_new", "model": "spatial_temporal_new", "run_dir": "runs/ood_spatial_temporal_new", "out_dir": "outputs/ood_spatial_temporal_new"},
    "R6": {"method": "stf3_new_concat", "model": "stf3_new_concat", "run_dir": "runs/ood_stf3_new_concat", "out_dir": "outputs/ood_stf3_new_concat"},
    "R7": {"method": "stf3_new", "model": "stf3_new", "run_dir": "runs/ood_stf3_new", "out_dir": "outputs/ood_stf3_new"},
}

def rel(path):
    path = Path(path)
    try:
        return path.resolve().relative_to(PROJECT_ROOT).as_posix()
    except Exception:
        return path.as_posix()

artifact_rows = []
for code, cfg in BASE_EXPERIMENTS.items():
    run_dir = PROJECT_ROOT / cfg["run_dir"]
    out_dir = PROJECT_ROOT / cfg["out_dir"]
    artifact_rows.append({
        "ID": code,
        "Method": cfg["method"],
        "best.pt": (run_dir / "best.pt").exists(),
        "history.json": (run_dir / "history.json").exists(),
        "test predictions": (out_dir / "predictions.csv").exists(),
        "val predictions": (PROJECT_ROOT / f"{cfg['out_dir']}_val" / "predictions.csv").exists(),
    })
display(pd.DataFrame(artifact_rows))

,ID,Method,best.pt,history.json,test predictions,val predictions
0,R1,frequency_wave,True,True,True,True
1,R2,spatial_dino,True,True,True,True
2,R3,temporal_d3_restrav,True,True,True,True
3,R4,spatial_frequency_new,True,True,True,True
4,R5,spatial_temporal_new,True,True,True,True
5,R6,stf3_new_concat,True,True,True,True
6,R7,stf3_new,True,True,True,True


## 3. 通用命令运行函数

日志会写入 `logs/ood_followup`。短分析命令会直接显示输出；长训练建议一次只跑一个 cell。

In [3]:
def run_cmd(cmd, desc, log_name=None, skip_if=None, force=False, compact_progress=True):
    if skip_if is not None and Path(skip_if).exists() and not force:
        print(f"[skip] {desc}: {rel(skip_if)} already exists")
        return 0
    log_name = log_name or desc.lower().replace(" ", "_").replace("/", "_") + ".log"
    log_path = LOG_DIR / log_name
    print(f"\n========== {desc} ==========")
    print("[cmd]", " ".join(str(x) for x in cmd))
    print("[log]", rel(log_path))
    start = datetime.now()

    progress_handle = None
    last_progress = ""
    last_update = start
    if compact_progress:
        progress_handle = display(Markdown("```text\nstarting...\n```"), display_id=True)

    def update_progress(text, force=False):
        nonlocal last_progress, last_update
        text = text.strip()
        if not text:
            return
        last_progress = text
        now = datetime.now()
        if progress_handle is not None and (force or (now - last_update).total_seconds() >= 1.0):
            elapsed_min = (now - start).total_seconds() / 60
            progress_handle.update(Markdown(f"```text\n{desc}, elapsed={elapsed_min:.1f} min\n{last_progress}\n```"))
            last_update = now

    with log_path.open("w", encoding="utf-8", errors="replace") as log:
        log.write(f"# {desc}\n# start: {start.isoformat(timespec='seconds')}\n")
        log.write("# cmd: " + " ".join(str(x) for x in cmd) + "\n\n")
        proc = subprocess.Popen(
            [str(x) for x in cmd],
            cwd=PROJECT_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )
        assert proc.stdout is not None
        buf = ""
        while True:
            ch = proc.stdout.read(1)
            if ch == "" and proc.poll() is not None:
                break
            if not ch:
                continue
            log.write(ch)
            if ch == "\r":
                if "%|" in buf or buf.strip().startswith(("predict", "epoch", "eval")):
                    update_progress(buf)
                buf = ""
            elif ch == "\n":
                line = buf.rstrip()
                buf = ""
                if not line.strip():
                    continue
                if "%|" in line or line.strip().startswith(("predict", "epoch", "eval")):
                    update_progress(line)
                else:
                    print(line)
            else:
                buf += ch
        if buf.strip():
            if "%|" in buf or buf.strip().startswith(("predict", "epoch", "eval")):
                update_progress(buf, force=True)
            else:
                print(buf.rstrip())
        rc = proc.wait()

    elapsed_min = (datetime.now() - start).total_seconds() / 60
    if rc != 0:
        raise RuntimeError(f"Command failed ({rc}): {desc}")
    if progress_handle is not None:
        done_line = last_progress or "completed"
        progress_handle.update(Markdown(f"```text\n{desc} done, elapsed={elapsed_min:.1f} min\n{done_line}\n```"))
    print(f"[done] {desc}, elapsed={elapsed_min:.1f} min")
    return rc

def show_csv(path, n=30):
    path = PROJECT_ROOT / path
    if not path.exists():
        print("[missing]", rel(path))
        return None
    df = pd.read_csv(path)
    display(df.head(n))
    print(f"rows={len(df)} path={rel(path)}")
    return df

## 4. Step A: 为 R1-R7 生成 val predictions

阈值必须在 val 上选择，不能在 test 上调。这里会用每个模型的 `best.pt` 在 `ood_val.csv` 上跑一次 `src.evaluate`，输出到 `outputs/ood_*_val`。

如果只想先做 R5/R7，把 `VAL_EVAL_IDS` 改成 `["R5", "R7"]`。

In [ ]:
VAL_EVAL_IDS = ["R1", "R2", "R3", "R4", "R5", "R6", "R7"]

for code in VAL_EVAL_IDS:
    cfg = BASE_EXPERIMENTS[code]
    ckpt = PROJECT_ROOT / cfg["run_dir"] / "best.pt"
    val_out = PROJECT_ROOT / f"{cfg['out_dir']}_val"
    if not ckpt.exists():
        print(f"[missing] {code} checkpoint: {rel(ckpt)}")
        continue
    cmd = [
        PYTHON, "-m", "src.evaluate",
        "--checkpoint", ckpt,
        "--csv", VAL_CSV,
        "--batch-size", "1",
        "--amp",
        "--out-dir", val_out,
    ]
    run_cmd(cmd, f"{code} val evaluate", log_name=f"{code.lower()}_{cfg['method']}_val_evaluate.log", skip_if=val_out / "predictions.csv", force=FORCE_EVAL)

## 5. Step B: Val-calibrated threshold results

脚本会在 val predictions 上找阈值，然后固定阈值重新计算 test predictions 的 ACC/F1/Precision/Recall。默认输出三种目标：

- `max_f1`
- `max_balanced_acc`
- `precision_0.95_recall_max`

主实验表仍保留 0.5 阈值，这张表作为后处理/校准补充。

In [ ]:
run_cmd([
    PYTHON, "scripts/ood_threshold_calibration.py",
    "--ids", ",".join(VAL_EVAL_IDS),
    "--out-dir", "outputs/ood_followup",
], "base threshold calibration", log_name="base_threshold_calibration.log", force=True)

threshold_df = show_csv("outputs/ood_followup/val_calibrated_thresholds.csv")

## 6. Step C: Generator-level 错误分析

先看默认 0.5 阈值，再看 val-calibrated 阈值。重点关注 fake 的 FN：哪些 generator 最容易漏检，以及 R7 是否修正 R5 的 FN。

In [ ]:
run_cmd([
    PYTHON, "scripts/ood_generator_error_analysis.py",
    "--ids", "R1,R2,R3,R4,R5,R6,R7",
    "--out-dir", "outputs/ood_followup",
], "generator error analysis default", log_name="generator_error_default.log", force=True)

show_csv("outputs/ood_followup/generator_error_analysis_default_0.5.csv")
show_csv("outputs/ood_followup/r5_r7_fn_comparison_default_0.5.csv")

In [ ]:
CALIBRATION_OBJECTIVE = "max_f1"  # 可改成 max_balanced_acc 或 precision_0.95_recall_max

run_cmd([
    PYTHON, "scripts/ood_generator_error_analysis.py",
    "--ids", "R1,R2,R3,R4,R5,R6,R7",
    "--thresholds-csv", "outputs/ood_followup/val_calibrated_thresholds.csv",
    "--objective", CALIBRATION_OBJECTIVE,
    "--out-dir", "outputs/ood_followup",
], f"generator error analysis calibrated {CALIBRATION_OBJECTIVE}", log_name=f"generator_error_{CALIBRATION_OBJECTIVE}.log", force=True)

show_csv(f"outputs/ood_followup/generator_error_analysis_{CALIBRATION_OBJECTIVE}.csv")
show_csv(f"outputs/ood_followup/r5_r7_fn_comparison_{CALIBRATION_OBJECTIVE}.csv")

## 7. Step D: R5 + R7 概率融合

这个实验不重训。做法是：在 val 上搜索 `w * p_R5 + (1 - w) * p_R7` 的 `w` 和阈值，再固定到 test。

In [ ]:
run_cmd([
    PYTHON, "scripts/ood_r5_r7_ensemble.py",
    "--weight-step", "0.05",
    "--out-dir", "outputs/ood_followup",
], "R5 R7 ensemble", log_name="r5_r7_ensemble.log", force=True)

ensemble_df = show_csv("outputs/ood_followup/r5_r7_ensemble_results.csv")

## 8. 可选长实验配置

下面这些实验会重新训练。默认不会运行，除非你把最上面的 `RUN_LONG_TRAINING` 改成 `True`。建议每次只运行一个长实验 cell，完成后先看结果，再决定是否继续。

In [4]:
COMMON_TRAIN_ARGS = [
    "--train-csv", TRAIN_CSV,
    "--val-csv", VAL_CSV,
    "--batch-size", "1",
    "--num-frames", "8",
    "--foundation-backbone", "dinov2_vits14",
    "--wavelet-aug-prob", "0.1",
    "--branch-dropout", "0.1",
    "--aux-loss-weight", "0.2",
    "--amp",
]

FOLLOWUP_EXPERIMENTS = {
    "R5_E6": {"method": "spatial_temporal_new_e6", "model": "spatial_temporal_new", "run_dir": "runs/ood_spatial_temporal_new_e6", "out_dir": "outputs/ood_spatial_temporal_new_e6", "epochs": 6, "image_size": 112, "fake_loss_weight": 1.0, "pos_weight": "none"},
    "R7_E6": {"method": "stf3_new_e6", "model": "stf3_new", "run_dir": "runs/ood_stf3_new_e6", "out_dir": "outputs/ood_stf3_new_e6", "epochs": 6, "image_size": 112, "fake_loss_weight": 1.0, "pos_weight": "none"},
    "R7_FAKEW12": {"method": "stf3_new_fakew12", "model": "stf3_new", "run_dir": "runs/ood_stf3_new_fakew12", "out_dir": "outputs/ood_stf3_new_fakew12", "epochs": 6, "image_size": 112, "fake_loss_weight": 1.2, "pos_weight": "none"},
    "R7_FAKEW15": {"method": "stf3_new_fakew15", "model": "stf3_new", "run_dir": "runs/ood_stf3_new_fakew15", "out_dir": "outputs/ood_stf3_new_fakew15", "epochs": 6, "image_size": 112, "fake_loss_weight": 1.5, "pos_weight": "none"},
    "R5_224": {"method": "spatial_temporal_new_224", "model": "spatial_temporal_new", "run_dir": "runs/ood_spatial_temporal_new_224", "out_dir": "outputs/ood_spatial_temporal_new_224", "epochs": 5, "image_size": 224, "fake_loss_weight": 1.0, "pos_weight": "none"},
    "R5_224_FAKEW12": {"method": "spatial_temporal_new_224_fakew12", "model": "spatial_temporal_new", "run_dir": "runs/ood_spatial_temporal_new_224_fakew12", "out_dir": "outputs/ood_spatial_temporal_new_224_fakew12", "epochs": 5, "image_size": 224, "fake_loss_weight": 1.2, "pos_weight": "none"},
    "R7_224": {"method": "stf3_new_224", "model": "stf3_new", "run_dir": "runs/ood_stf3_new_224", "out_dir": "outputs/ood_stf3_new_224", "epochs": 5, "image_size": 224, "fake_loss_weight": 1.0, "pos_weight": "none"},
    "R7_224_FAKEW12": {"method": "stf3_new_224_fakew12", "model": "stf3_new", "run_dir": "runs/ood_stf3_new_224_fakew12", "out_dir": "outputs/ood_stf3_new_224_fakew12", "epochs": 5, "image_size": 224, "fake_loss_weight": 1.2, "pos_weight": "none"},
}

FOLLOWUP_JSON = OUT_DIR / "followup_experiments.json"
FOLLOWUP_JSON.write_text(json.dumps({k: {"method": v["method"], "out_dir": v["out_dir"]} for k, v in FOLLOWUP_EXPERIMENTS.items()}, ensure_ascii=False, indent=2), encoding="utf-8")
print("[write]", rel(FOLLOWUP_JSON))
display(pd.DataFrame([{"ID": k, **v} for k, v in FOLLOWUP_EXPERIMENTS.items()]))

[write] outputs/ood_followup/followup_experiments.json


,ID,method,model,run_dir,out_dir,epochs,image_size,fake_loss_weight,pos_weight
0,R5_E6,spatial_temporal_new_e6,spatial_temporal_new,runs/ood_spatial_temporal_new_e6,outputs/ood_spatial_temporal_new_e6,6,112,1.0,none
1,R7_E6,stf3_new_e6,stf3_new,runs/ood_stf3_new_e6,outputs/ood_stf3_new_e6,6,112,1.0,none
2,R7_FAKEW12,stf3_new_fakew12,stf3_new,runs/ood_stf3_new_fakew12,outputs/ood_stf3_new_fakew12,6,112,1.2,none
3,R7_FAKEW15,stf3_new_fakew15,stf3_new,runs/ood_stf3_new_fakew15,outputs/ood_stf3_new_fakew15,6,112,1.5,none
4,R5_224,spatial_temporal_new_224,spatial_temporal_new,runs/ood_spatial_temporal_new_224,outputs/ood_spatial_temporal_new_224,5,224,1.0,none
5,R5_224_FAKEW12,spatial_temporal_new_224_fakew12,spatial_temporal_new,runs/ood_spatial_temporal_new_224_fakew12,outputs/ood_spatial_temporal_new_224_fakew12,5,224,1.2,none
6,R7_224,stf3_new_224,stf3_new,runs/ood_stf3_new_224,outputs/ood_stf3_new_224,5,224,1.0,none
7,R7_224_FAKEW12,stf3_new_224_fakew12,stf3_new,runs/ood_stf3_new_224_fakew12,outputs/ood_stf3_new_224_fakew12,5,224,1.2,none


In [5]:
def train_eval_followup(exp_id, force=False):
    cfg = FOLLOWUP_EXPERIMENTS[exp_id]
    run_dir = PROJECT_ROOT / cfg["run_dir"]
    out_dir = PROJECT_ROOT / cfg["out_dir"]
    val_out_dir = PROJECT_ROOT / f"{cfg['out_dir']}_val"
    if not RUN_LONG_TRAINING:
        print(f"[guard] RUN_LONG_TRAINING=False, skip {exp_id}. Set it to True only when you are ready to run this one experiment.")
        return

    train_cmd = [
        PYTHON, "-m", "src.train",
        "--model", cfg["model"],
        "--epochs", str(cfg["epochs"]),
        "--image-size", str(cfg["image_size"]),
        "--fake-loss-weight", str(cfg["fake_loss_weight"]),
        "--pos-weight", cfg["pos_weight"],
        "--out-dir", run_dir,
        *COMMON_TRAIN_ARGS,
    ]
    run_cmd(train_cmd, f"{exp_id} train", log_name=f"{exp_id.lower()}_train.log", skip_if=run_dir / "best.pt", force=force)

    eval_cmd = [
        PYTHON, "-m", "src.evaluate",
        "--checkpoint", run_dir / "best.pt",
        "--csv", TEST_CSV,
        "--batch-size", "1",
        "--amp",
        "--out-dir", out_dir,
    ]
    run_cmd(eval_cmd, f"{exp_id} test evaluate", log_name=f"{exp_id.lower()}_test_evaluate.log", skip_if=out_dir / "predictions.csv", force=force or FORCE_EVAL)

    val_cmd = [
        PYTHON, "-m", "src.evaluate",
        "--checkpoint", run_dir / "best.pt",
        "--csv", VAL_CSV,
        "--batch-size", "1",
        "--amp",
        "--out-dir", val_out_dir,
    ]
    run_cmd(val_cmd, f"{exp_id} val evaluate", log_name=f"{exp_id.lower()}_val_evaluate.log", skip_if=val_out_dir / "predictions.csv", force=force or FORCE_EVAL)

    print(f"[ready] {exp_id}: run follow-up calibration cell after this.")

### 8.1 R5 spatial_temporal_new, 6 epoch

用于检查 R5 的高 AUC 是否能通过更充分训练转成更高 Recall/F1。

In [ ]:
train_eval_followup("R5_E6")

### 8.2 R7 stf3_new, 6 epoch

用于检查 R7 的默认阈值稳定性是否还能继续提升。

In [ ]:
train_eval_followup("R7_E6")

### 8.3 R7 fake loss weight = 1.2

召回率导向训练。观察 Recall 是否上升，以及 Precision 是否明显下降。

In [ ]:
train_eval_followup("R7_FAKEW12", force=True)

### 8.4 R7 fake loss weight = 1.5

比 1.2 更激进。若 Precision 下滑明显，不建议作为主模型。

In [ ]:
train_eval_followup("R7_FAKEW15")

### 8.5 R5 image-size = 224

只作为重点验证，不建议全模型重跑 224。

In [ ]:
train_eval_followup("R5_224")

### 8.6 R7 image-size = 224

如果 224 明显提升 WildScrape/Sora Recall，才值得写入论文补充实验。

In [ ]:
train_eval_followup("R7_224", force=True)

### 8.7 R7 image-size = 224 + fake loss weight = 1.2

这个补充实验组合两个已观察到的方向：`224` 输入提升整体 OOD 表现，`fake_loss_weight=1.2` 对 WildScrape 有小幅帮助。运行后仍然需要第 9 模块做 val-calibrated threshold 和 generator-level 分析。

In [6]:
train_eval_followup("R7_224_FAKEW12")


========== R7_224_FAKEW12 train ==========
[cmd] D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe -m src.train --model stf3_new --epochs 5 --image-size 224 --fake-loss-weight 1.2 --pos-weight none --out-dir D:\VsCode Program\Python\content_security\final_project\runs\ood_stf3_new_224_fakew12 --train-csv data/GenVideo-Val/splits/ood_train.csv --val-csv data/GenVideo-Val/splits/ood_val.csv --batch-size 1 --num-frames 8 --foundation-backbone dinov2_vits14 --wavelet-aug-prob 0.1 --branch-dropout 0.1 --aux-loss-weight 0.2 --amp
[log] logs/ood_followup/r7_224_fakew12_train.log


```text
R7_224_FAKEW12 train done, elapsed=275.8 min
eval: 100%|█████████▉| 2428/2429 [07:18<00:00,  9.12it/s]
```

[device] cuda
[model] stf3_new
Using cache found in C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
D:\VsCode Program\Python\content_security\final_project\src\models\fusion\branch_token_transformer.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
[loss] c

```text
R7_224_FAKEW12 test evaluate done, elapsed=10.9 min
predict: 100%|██████████| 3598/3598 [10:39<00:00,  5.63it/s]
```

Using cache found in C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
D:\VsCode Program\Python\content_security\final_project\src\models\fusion\branch_token_transformer.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
{'acc': 0.9274596998332407, 'auc': 0.98

```text
R7_224_FAKEW12 val evaluate done, elapsed=7.4 min
predict: 100%|██████████| 2429/2429 [07:16<00:00,  5.57it/s]
```

Using cache found in C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
D:\VsCode Program\Python\content_security\final_project\src\models\fusion\branch_token_transformer.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
{'acc': 0.9897076986414162, 'auc': 0.99

### 8.8 R5 image-size = 224 + fake loss weight = 1.2

这是最终优化公平对照实验。它与 `R7_224_FAKEW12` 使用相同的分辨率、epoch、fake loss weight、OOD train/val/test 和阈值校准流程，差异只在模型结构：`R5 = spatial + temporal`，`R7 = spatial + temporal + frequency`。


In [6]:
FOLLOWUP_EXPERIMENTS["R5_224_FAKEW12"] = {
    "method": "spatial_temporal_new_224_fakew12",
    "model": "spatial_temporal_new",
    "run_dir": "runs/ood_spatial_temporal_new_224_fakew12",
    "out_dir": "outputs/ood_spatial_temporal_new_224_fakew12",
    "epochs": 5,
    "image_size": 224,
    "fake_loss_weight": 1.2,
    "pos_weight": "none",
}
FOLLOWUP_JSON.write_text(
    json.dumps({k: {"method": v["method"], "out_dir": v["out_dir"]} for k, v in FOLLOWUP_EXPERIMENTS.items()}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("[update]", rel(FOLLOWUP_JSON))
train_eval_followup("R5_224_FAKEW12")


[update] outputs/ood_followup/followup_experiments.json

========== R5_224_FAKEW12 train ==========
[cmd] D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe -m src.train --model spatial_temporal_new --epochs 5 --image-size 224 --fake-loss-weight 1.2 --pos-weight none --out-dir D:\VsCode Program\Python\content_security\final_project\runs\ood_spatial_temporal_new_224_fakew12 --train-csv data/GenVideo-Val/splits/ood_train.csv --val-csv data/GenVideo-Val/splits/ood_val.csv --batch-size 1 --num-frames 8 --foundation-backbone dinov2_vits14 --wavelet-aug-prob 0.1 --branch-dropout 0.1 --aux-loss-weight 0.2 --amp
[log] logs/ood_followup/r5_224_fakew12_train.log


```text
R5_224_FAKEW12 train done, elapsed=241.4 min
eval: 100%|██████████| 2429/2429 [07:29<00:00,  8.66it/s]
```

[device] cuda
[model] spatial_temporal_new
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
D:\VsCode Program\Python\content_security\final_project\src\models\fusion\branch_token_transformer.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
[loss] class_weight real=1.0000 fake=1.2000
[params] trainable=1,822,461 total

```text
R5_224_FAKEW12 test evaluate done, elapsed=11.3 min
predict: 100%|██████████| 3598/3598 [11:07<00:00,  5.39it/s]
```

C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
D:\VsCode Program\Python\content_security\final_project\src\models\fusion\branch_token_transformer.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
{'acc': 0.8996664813785437, 'auc': 0.9801973307912298, 'f1': 0.9062094050402703, 'precision': 0.9960022844089091, 'recall

```text
R5_224_FAKEW12 val evaluate done, elapsed=7.7 min
predict: 100%|██████████| 2429/2429 [07:36<00:00,  5.33it/s]
```

C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
D:\VsCode Program\Python\content_security\final_project\src\models\fusion\branch_token_transformer.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
{'acc': 0.9876492383696994, 'auc': 0.9988302834589164, 'f1': 0.9837662337662337, 'precision': 0.9891186071817193, 'recall

## 9. 长实验跑完后的统一校准与错误分析

每完成一个可选长实验，都可以运行这一节。缺少 predictions 的实验会自动跳过。

In [7]:
FOLLOWUP_IDS = ["R5_E6", "R7_E6", "R7_FAKEW12", "R7_FAKEW15", "R5_224", "R5_224_FAKEW12", "R7_224", "R7_224_FAKEW12"]

run_cmd([
    PYTHON, "scripts/ood_threshold_calibration.py",
    "--ids", ",".join(FOLLOWUP_IDS),
    "--extra-experiments-json", rel(FOLLOWUP_JSON),
    "--out-dir", "outputs/ood_followup/followup_runs",
], "followup threshold calibration", log_name="followup_threshold_calibration.log", force=True)

show_csv("outputs/ood_followup/followup_runs/val_calibrated_thresholds.csv")


========== followup threshold calibration ==========
[cmd] D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe scripts/ood_threshold_calibration.py --ids R5_E6,R7_E6,R7_FAKEW12,R7_FAKEW15,R5_224,R5_224_FAKEW12,R7_224,R7_224_FAKEW12 --extra-experiments-json outputs/ood_followup/followup_experiments.json --out-dir outputs/ood_followup/followup_runs
[log] logs/ood_followup/followup_threshold_calibration.log


```text
followup threshold calibration done, elapsed=0.4 min
completed
```

[skip] R5_E6 missing predictions: val=False test=False
[skip] R7_FAKEW15 missing predictions: val=False test=False
[skip] R5_224 missing predictions: val=False test=False
[write] outputs\ood_followup\followup_runs\val_calibrated_thresholds.csv
[write] outputs\ood_followup\followup_runs\val_calibrated_thresholds.md
[write] outputs\ood_followup\followup_runs\val_calibrated_thresholds.json
[done] followup threshold calibration, elapsed=0.4 min


,id,method,objective,threshold,val_acc,val_f1,val_precision,val_recall,val_balanced_acc,val_fn,...,test_recall,test_balanced_acc,test_fn,test_fp,default_test_acc,default_test_f1,default_test_precision,default_test_recall,default_test_fn,default_test_fp
0,R7_E6,stf3_new_e6,max_f1,0.112793,0.980239,0.974304,0.969116,0.979548,0.980107,19,...,0.867016,0.927508,279,18,0.896609,0.903226,0.994273,0.827455,362,10
1,R7_E6,stf3_new_e6,max_balanced_acc,0.112793,0.980239,0.974304,0.969116,0.979548,0.980107,19,...,0.867016,0.927508,279,18,0.896609,0.903226,0.994273,0.827455,362,10
2,R7_E6,stf3_new_e6,precision_0.95_recall_max,0.017303,0.976534,0.969953,0.950413,0.990312,0.979156,9,...,0.895138,0.934902,220,38,0.896609,0.903226,0.994273,0.827455,362,10
3,R7_E6,stf3_new_e6,max_acc,0.112793,0.980239,0.974304,0.969116,0.979548,0.980107,19,...,0.867016,0.927508,279,18,0.896609,0.903226,0.994273,0.827455,362,10
4,R7_FAKEW12,stf3_new_fakew12,max_f1,0.569336,0.979827,0.973499,0.978261,0.968784,0.977725,29,...,0.857007,0.922837,300,17,0.916342,0.923468,0.989646,0.865586,282,19
5,R7_FAKEW12,stf3_new_fakew12,max_balanced_acc,0.315186,0.979415,0.973433,0.961175,0.986006,0.980670,13,...,0.887512,0.934089,236,29,0.916342,0.923468,0.989646,0.865586,282,19
6,R7_FAKEW12,stf3_new_fakew12,precision_0.95_recall_max,0.245728,0.977357,0.971007,0.951446,0.991389,0.980028,8,...,0.894662,0.935997,221,34,0.916342,0.923468,0.989646,0.865586,282,19
7,R7_FAKEW12,stf3_new_fakew12,max_acc,0.569336,0.979827,0.973499,0.978261,0.968784,0.977725,29,...,0.857007,0.922837,300,17,0.916342,0.923468,0.989646,0.865586,282,19
8,R5_224_FAKEW12,spatial_temporal_new_224_fakew12,max_f1,0.161743,0.991354,0.988691,0.989224,0.988159,0.990746,11,...,0.862250,0.927792,289,10,0.899666,0.906209,0.996002,0.831268,354,7
9,R5_224_FAKEW12,spatial_temporal_new_224_fakew12,max_balanced_acc,0.161743,0.991354,0.988691,0.989224,0.988159,0.990746,11,...,0.862250,0.927792,289,10,0.899666,0.906209,0.996002,0.831268,354,7


rows=20 path=outputs/ood_followup/followup_runs/val_calibrated_thresholds.csv


,id,method,objective,threshold,val_acc,val_f1,val_precision,val_recall,val_balanced_acc,val_fn,...,test_recall,test_balanced_acc,test_fn,test_fp,default_test_acc,default_test_f1,default_test_precision,default_test_recall,default_test_fn,default_test_fp
0,R7_E6,stf3_new_e6,max_f1,0.112793,0.980239,0.974304,0.969116,0.979548,0.980107,19,...,0.867016,0.927508,279,18,0.896609,0.903226,0.994273,0.827455,362,10
1,R7_E6,stf3_new_e6,max_balanced_acc,0.112793,0.980239,0.974304,0.969116,0.979548,0.980107,19,...,0.867016,0.927508,279,18,0.896609,0.903226,0.994273,0.827455,362,10
2,R7_E6,stf3_new_e6,precision_0.95_recall_max,0.017303,0.976534,0.969953,0.950413,0.990312,0.979156,9,...,0.895138,0.934902,220,38,0.896609,0.903226,0.994273,0.827455,362,10
3,R7_E6,stf3_new_e6,max_acc,0.112793,0.980239,0.974304,0.969116,0.979548,0.980107,19,...,0.867016,0.927508,279,18,0.896609,0.903226,0.994273,0.827455,362,10
4,R7_FAKEW12,stf3_new_fakew12,max_f1,0.569336,0.979827,0.973499,0.978261,0.968784,0.977725,29,...,0.857007,0.922837,300,17,0.916342,0.923468,0.989646,0.865586,282,19
5,R7_FAKEW12,stf3_new_fakew12,max_balanced_acc,0.315186,0.979415,0.973433,0.961175,0.986006,0.980670,13,...,0.887512,0.934089,236,29,0.916342,0.923468,0.989646,0.865586,282,19
6,R7_FAKEW12,stf3_new_fakew12,precision_0.95_recall_max,0.245728,0.977357,0.971007,0.951446,0.991389,0.980028,8,...,0.894662,0.935997,221,34,0.916342,0.923468,0.989646,0.865586,282,19
7,R7_FAKEW12,stf3_new_fakew12,max_acc,0.569336,0.979827,0.973499,0.978261,0.968784,0.977725,29,...,0.857007,0.922837,300,17,0.916342,0.923468,0.989646,0.865586,282,19
8,R5_224_FAKEW12,spatial_temporal_new_224_fakew12,max_f1,0.161743,0.991354,0.988691,0.989224,0.988159,0.990746,11,...,0.862250,0.927792,289,10,0.899666,0.906209,0.996002,0.831268,354,7
9,R5_224_FAKEW12,spatial_temporal_new_224_fakew12,max_balanced_acc,0.161743,0.991354,0.988691,0.989224,0.988159,0.990746,11,...,0.862250,0.927792,289,10,0.899666,0.906209,0.996002,0.831268,354,7


In [8]:
FOLLOWUP_CALIBRATION_OBJECTIVE = "precision_0.95_recall_max"

run_cmd([
    PYTHON, "scripts/ood_generator_error_analysis.py",
    "--ids", ",".join(FOLLOWUP_IDS),
    "--extra-experiments-json", rel(FOLLOWUP_JSON),
    "--thresholds-csv", "outputs/ood_followup/followup_runs/val_calibrated_thresholds.csv",
    "--objective", FOLLOWUP_CALIBRATION_OBJECTIVE,
    "--out-dir", "outputs/ood_followup/followup_runs",
], "followup generator analysis", log_name="followup_generator_analysis.log", force=True)

show_csv(f"outputs/ood_followup/followup_runs/generator_error_analysis_{FOLLOWUP_CALIBRATION_OBJECTIVE}.csv")


========== followup generator analysis ==========
[cmd] D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe scripts/ood_generator_error_analysis.py --ids R5_E6,R7_E6,R7_FAKEW12,R7_FAKEW15,R5_224,R5_224_FAKEW12,R7_224,R7_224_FAKEW12 --extra-experiments-json outputs/ood_followup/followup_experiments.json --thresholds-csv outputs/ood_followup/followup_runs/val_calibrated_thresholds.csv --objective precision_0.95_recall_max --out-dir outputs/ood_followup/followup_runs
[log] logs/ood_followup/followup_generator_analysis.log


```text
followup generator analysis done, elapsed=0.0 min
completed
```

[skip] R5_E6 missing outputs\ood_spatial_temporal_new_e6\predictions.csv
[skip] R7_FAKEW15 missing outputs\ood_stf3_new_fakew15\predictions.csv
[skip] R5_224 missing outputs\ood_spatial_temporal_new_224\predictions.csv
[write] outputs\ood_followup\followup_runs\generator_error_analysis_precision_0.95_recall_max.csv
[write] outputs\ood_followup\followup_runs\r5_r7_fn_comparison_precision_0.95_recall_max.csv
[write] outputs\ood_followup\followup_runs\generator_error_analysis_precision_0.95_recall_max.md
[done] followup generator analysis, elapsed=0.0 min


,generator,n,real_n,fake_n,acc,precision,recall,specificity,tn,fp,fn,tp,id,method,threshold
0,MorphStudio,700,0,700,0.985714,1.0,0.985714,NaN,0,0,10,690,R7_E6,stf3_new_e6,0.017303
1,Show_1,700,0,700,0.962857,1.0,0.962857,NaN,0,0,26,674,R7_E6,stf3_new_e6,0.017303
2,Sora,56,0,56,0.857143,1.0,0.857143,NaN,0,0,8,48,R7_E6,stf3_new_e6,0.017303
3,WildScrape,642,0,642,0.725857,1.0,0.725857,NaN,0,0,176,466,R7_E6,stf3_new_e6,0.017303
4,real_MSRVTT,1500,1500,0,0.974667,0.0,NaN,0.974667,1462,38,0,0,R7_E6,stf3_new_e6,0.017303
5,MorphStudio,700,0,700,0.971429,1.0,0.971429,NaN,0,0,20,680,R7_FAKEW12,stf3_new_fakew12,0.245728
6,Show_1,700,0,700,0.967143,1.0,0.967143,NaN,0,0,23,677,R7_FAKEW12,stf3_new_fakew12,0.245728
7,Sora,56,0,56,0.821429,1.0,0.821429,NaN,0,0,10,46,R7_FAKEW12,stf3_new_fakew12,0.245728
8,WildScrape,642,0,642,0.738318,1.0,0.738318,NaN,0,0,168,474,R7_FAKEW12,stf3_new_fakew12,0.245728
9,real_MSRVTT,1500,1500,0,0.977333,0.0,NaN,0.977333,1466,34,0,0,R7_FAKEW12,stf3_new_fakew12,0.245728


rows=25 path=outputs/ood_followup/followup_runs/generator_error_analysis_precision_0.95_recall_max.csv


,generator,n,real_n,fake_n,acc,precision,recall,specificity,tn,fp,fn,tp,id,method,threshold
0,MorphStudio,700,0,700,0.985714,1.0,0.985714,NaN,0,0,10,690,R7_E6,stf3_new_e6,0.017303
1,Show_1,700,0,700,0.962857,1.0,0.962857,NaN,0,0,26,674,R7_E6,stf3_new_e6,0.017303
2,Sora,56,0,56,0.857143,1.0,0.857143,NaN,0,0,8,48,R7_E6,stf3_new_e6,0.017303
3,WildScrape,642,0,642,0.725857,1.0,0.725857,NaN,0,0,176,466,R7_E6,stf3_new_e6,0.017303
4,real_MSRVTT,1500,1500,0,0.974667,0.0,NaN,0.974667,1462,38,0,0,R7_E6,stf3_new_e6,0.017303
5,MorphStudio,700,0,700,0.971429,1.0,0.971429,NaN,0,0,20,680,R7_FAKEW12,stf3_new_fakew12,0.245728
6,Show_1,700,0,700,0.967143,1.0,0.967143,NaN,0,0,23,677,R7_FAKEW12,stf3_new_fakew12,0.245728
7,Sora,56,0,56,0.821429,1.0,0.821429,NaN,0,0,10,46,R7_FAKEW12,stf3_new_fakew12,0.245728
8,WildScrape,642,0,642,0.738318,1.0,0.738318,NaN,0,0,168,474,R7_FAKEW12,stf3_new_fakew12,0.245728
9,real_MSRVTT,1500,1500,0,0.977333,0.0,NaN,0.977333,1466,34,0,0,R7_FAKEW12,stf3_new_fakew12,0.245728


## 10. 不重训的高潜力优化实验

这一节只围绕当前主模型 `R7_224_FAKEW12` 做后处理和推理优化，不重新训练主模型。建议按顺序逐个运行，不要 Run All。

本节新增 5 个实验：

1. `max_acc` 阈值搜索。
2. `R7_224_FAKEW12_5CLIP_MEAN`。
3. `R7_224_FAKEW12_5CLIP_TOP2_MEAN`。
4. `R7_224_FAKEW12_PLUS_R7_224`。
5. `R7_224_FAKEW12_BRANCH_CALIBRATION_LR`。


In [8]:
OPT_DIR = OUT_DIR / "optimization_runs"
OPT_DIR.mkdir(parents=True, exist_ok=True)

MAIN_R7_ID = "R7_224_FAKEW12"
MAIN_R7_CFG = FOLLOWUP_EXPERIMENTS[MAIN_R7_ID]
MAIN_R7_CKPT = PROJECT_ROOT / MAIN_R7_CFG["run_dir"] / "best.pt"

if not MAIN_R7_CKPT.exists():
    raise FileNotFoundError(f"Missing main checkpoint: {MAIN_R7_CKPT}. 请先完成 8.7。")

OPTIMIZATION_EXPERIMENTS = {
    "R7_224_FAKEW12_5CLIP_MEAN": {
        "method": "stf3_new_224_fakew12_5clip_mean",
        "out_dir": "outputs/ood_followup/optimization_runs/r7_224_fakew12_5clip_mean",
    },
    "R7_224_FAKEW12_5CLIP_TOP2_MEAN": {
        "method": "stf3_new_224_fakew12_5clip_top2_mean",
        "out_dir": "outputs/ood_followup/optimization_runs/r7_224_fakew12_5clip_top2_mean",
    },
}
OPTIMIZATION_JSON = OUT_DIR / "optimization_experiments.json"
OPTIMIZATION_JSON.write_text(json.dumps(OPTIMIZATION_EXPERIMENTS, ensure_ascii=False, indent=2), encoding="utf-8")

print("MAIN_R7_CKPT =", rel(MAIN_R7_CKPT))
print("OPT_DIR      =", rel(OPT_DIR))
print("OPT_JSON     =", rel(OPTIMIZATION_JSON))


MAIN_R7_CKPT = runs/ood_stf3_new_224_fakew12/best.pt
OPT_DIR      = outputs/ood_followup/optimization_runs
OPT_JSON     = outputs/ood_followup/optimization_experiments.json


### 10.1 `max_acc` 阈值搜索

这里会重新运行 followup threshold calibration。脚本已加入 `max_acc`，所以表格里会多出 `objective = max_acc` 的行。

这个实验不产生新 predictions，只是在已有 val/test predictions 上重新选阈值。


In [9]:
FOLLOWUP_IDS = ["R5_E6", "R7_E6", "R7_FAKEW12", "R7_FAKEW15", "R5_224", "R5_224_FAKEW12", "R7_224", "R7_224_FAKEW12"]

run_cmd([
    PYTHON, "scripts/ood_threshold_calibration.py",
    "--ids", ",".join(FOLLOWUP_IDS),
    "--extra-experiments-json", rel(FOLLOWUP_JSON),
    "--out-dir", "outputs/ood_followup/followup_runs",
], "followup threshold calibration with max_acc", log_name="followup_threshold_calibration_maxacc.log", force=True)

threshold_df = show_csv("outputs/ood_followup/followup_runs/val_calibrated_thresholds.csv", n=80)
if threshold_df is not None:
    display(threshold_df[(threshold_df["id"] == "R7_224_FAKEW12") & (threshold_df["objective"].isin(["max_acc", "precision_0.95_recall_max", "max_f1", "max_balanced_acc"]))])



========== followup threshold calibration with max_acc ==========
[cmd] D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe scripts/ood_threshold_calibration.py --ids R5_E6,R7_E6,R7_FAKEW12,R7_FAKEW15,R5_224,R7_224,R7_224_FAKEW12 --extra-experiments-json outputs/ood_followup/followup_experiments.json --out-dir outputs/ood_followup/followup_runs
[log] logs/ood_followup/followup_threshold_calibration_maxacc.log


```text
followup threshold calibration with max_acc done, elapsed=0.3 min
completed
```

[skip] R5_E6 missing predictions: val=False test=False
[skip] R7_FAKEW15 missing predictions: val=False test=False
[skip] R5_224 missing predictions: val=False test=False
[write] outputs\ood_followup\followup_runs\val_calibrated_thresholds.csv
[write] outputs\ood_followup\followup_runs\val_calibrated_thresholds.md
[write] outputs\ood_followup\followup_runs\val_calibrated_thresholds.json
[done] followup threshold calibration with max_acc, elapsed=0.3 min


,id,method,objective,threshold,val_acc,val_f1,val_precision,val_recall,val_balanced_acc,val_fn,...,test_recall,test_balanced_acc,test_fn,test_fp,default_test_acc,default_test_f1,default_test_precision,default_test_recall,default_test_fn,default_test_fp
0,R7_E6,stf3_new_e6,max_f1,0.112793,0.980239,0.974304,0.969116,0.979548,0.980107,19,...,0.867016,0.927508,279,18,0.896609,0.903226,0.994273,0.827455,362,10
1,R7_E6,stf3_new_e6,max_balanced_acc,0.112793,0.980239,0.974304,0.969116,0.979548,0.980107,19,...,0.867016,0.927508,279,18,0.896609,0.903226,0.994273,0.827455,362,10
2,R7_E6,stf3_new_e6,precision_0.95_recall_max,0.017303,0.976534,0.969953,0.950413,0.990312,0.979156,9,...,0.895138,0.934902,220,38,0.896609,0.903226,0.994273,0.827455,362,10
3,R7_E6,stf3_new_e6,max_acc,0.112793,0.980239,0.974304,0.969116,0.979548,0.980107,19,...,0.867016,0.927508,279,18,0.896609,0.903226,0.994273,0.827455,362,10
4,R7_FAKEW12,stf3_new_fakew12,max_f1,0.569336,0.979827,0.973499,0.978261,0.968784,0.977725,29,...,0.857007,0.922837,300,17,0.916342,0.923468,0.989646,0.865586,282,19
5,R7_FAKEW12,stf3_new_fakew12,max_balanced_acc,0.315186,0.979415,0.973433,0.961175,0.986006,0.980670,13,...,0.887512,0.934089,236,29,0.916342,0.923468,0.989646,0.865586,282,19
6,R7_FAKEW12,stf3_new_fakew12,precision_0.95_recall_max,0.245728,0.977357,0.971007,0.951446,0.991389,0.980028,8,...,0.894662,0.935997,221,34,0.916342,0.923468,0.989646,0.865586,282,19
7,R7_FAKEW12,stf3_new_fakew12,max_acc,0.569336,0.979827,0.973499,0.978261,0.968784,0.977725,29,...,0.857007,0.922837,300,17,0.916342,0.923468,0.989646,0.865586,282,19
8,R7_224,stf3_new_224,max_f1,0.524902,0.988884,0.985398,0.990217,0.980624,0.987312,18,...,0.837464,0.916399,341,7,0.904391,0.911019,0.996041,0.839371,337,7
9,R7_224,stf3_new_224,max_balanced_acc,0.056122,0.988061,0.984467,0.979744,0.989236,0.988285,10,...,0.878932,0.933466,254,18,0.904391,0.911019,0.996041,0.839371,337,7


rows=16 path=outputs/ood_followup/followup_runs/val_calibrated_thresholds.csv


,id,method,objective,threshold,val_acc,val_f1,val_precision,val_recall,val_balanced_acc,val_fn,...,test_recall,test_balanced_acc,test_fn,test_fp,default_test_acc,default_test_f1,default_test_precision,default_test_recall,default_test_fn,default_test_fp
12,R7_224_FAKEW12,stf3_new_224_fakew12,max_f1,0.250732,0.990119,0.987179,0.979852,0.994618,0.990976,5,...,0.905148,0.943574,199,27,0.92746,0.934108,0.993022,0.881792,248,13
13,R7_224_FAKEW12,stf3_new_224_fakew12,max_balanced_acc,0.250732,0.990119,0.987179,0.979852,0.994618,0.990976,5,...,0.905148,0.943574,199,27,0.92746,0.934108,0.993022,0.881792,248,13
14,R7_224_FAKEW12,stf3_new_224_fakew12,precision_0.95_recall_max,0.054459,0.981474,0.976303,0.955670,0.997847,0.984590,2,...,0.927073,0.946203,153,52,0.92746,0.934108,0.993022,0.881792,248,13
15,R7_224_FAKEW12,stf3_new_224_fakew12,max_acc,0.250732,0.990119,0.987179,0.979852,0.994618,0.990976,5,...,0.905148,0.943574,199,27,0.92746,0.934108,0.993022,0.881792,248,13


### 10.2 Multi-clip 推理函数

这个函数会分别在 `ood_val.csv` 和 `ood_test.csv` 上跑 multi-clip predictions。运行时间会明显长于普通 evaluate，因为每个视频要读 5 个 clip。

输出仍然是标准的：

```text
metrics.json
predictions.csv
```


In [18]:
def run_multiclip_optimization(exp_id, force=False):
    cfg = OPTIMIZATION_EXPERIMENTS[exp_id]
    out_dir = PROJECT_ROOT / cfg["out_dir"]
    val_out_dir = PROJECT_ROOT / f"{cfg['out_dir']}_val"

    if exp_id.endswith("5CLIP_MEAN"):
        aggregate = "mean"
        topk = "2"
    elif exp_id.endswith("5CLIP_TOP2_MEAN"):
        aggregate = "topk_mean"
        topk = "2"
    else:
        raise ValueError(f"Unknown multi-clip exp_id: {exp_id}")

    common = [
        PYTHON, "scripts/ood_multiclip_predict.py",
        "--checkpoint", MAIN_R7_CKPT,
        "--batch-size", "1",
        "--clip-count", "5",
        "--clip-crop-ratio", "0.6",
        "--aggregate", aggregate,
        "--topk", topk,
        "--amp",
    ]

    run_cmd([
        *common,
        "--csv", TEST_CSV,
        "--out-dir", out_dir,
    ], f"{exp_id} test multi-clip", log_name=f"{exp_id.lower()}_test_multiclip.log", skip_if=out_dir / "predictions.csv", force=force or FORCE_EVAL)

    run_cmd([
        *common,
        "--csv", VAL_CSV,
        "--out-dir", val_out_dir,
    ], f"{exp_id} val multi-clip", log_name=f"{exp_id.lower()}_val_multiclip.log", skip_if=val_out_dir / "predictions.csv", force=force or FORCE_EVAL)

    print(f"[ready] {exp_id}: 运行 10.5 做 multi-clip 阈值校准与 generator 分析。")


### 10.3 `R7_224_FAKEW12_5CLIP_MEAN`

每个视频采 5 个 clip，取 5 个 fake probability 的平均值。


In [19]:
run_multiclip_optimization("R7_224_FAKEW12_5CLIP_MEAN")


========== R7_224_FAKEW12_5CLIP_MEAN test multi-clip ==========
[cmd] D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe scripts/ood_multiclip_predict.py --checkpoint D:\VsCode Program\Python\content_security\final_project\runs\ood_stf3_new_224_fakew12\best.pt --batch-size 1 --clip-count 5 --clip-crop-ratio 0.6 --aggregate mean --topk 2 --amp --csv data/GenVideo-Val/splits/ood_test.csv --out-dir D:\VsCode Program\Python\content_security\final_project\outputs\ood_followup\optimization_runs\r7_224_fakew12_5clip_mean
[log] logs/ood_followup/r7_224_fakew12_5clip_mean_test_multiclip.log


```text
R7_224_FAKEW12_5CLIP_MEAN test multi-clip done, elapsed=41.4 min
predict: 100%|██████████| 3598/3598 [41:04<00:00,  1.46it/s]
```

Using cache found in C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
D:\VsCode Program\Python\content_security\final_project\src\models\fusion\branch_token_transformer.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
{'acc': 0.925236242356865, 'auc': 0.979

```text
R7_224_FAKEW12_5CLIP_MEAN val multi-clip done, elapsed=30.5 min
predict: 100%|██████████| 2429/2429 [30:18<00:00,  1.34it/s]
```

Using cache found in C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
D:\VsCode Program\Python\content_security\final_project\src\models\fusion\branch_token_transformer.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
{'acc': 0.984767393989296, 'auc': 0.999

### 10.4 `R7_224_FAKEW12_5CLIP_TOP2_MEAN`

每个视频采 5 个 clip，只取 fake probability 最高的 2 个 clip 求平均。这个实验更激进，可能降低 FN，也可能增加 FP。


In [20]:
run_multiclip_optimization("R7_224_FAKEW12_5CLIP_TOP2_MEAN")


========== R7_224_FAKEW12_5CLIP_TOP2_MEAN test multi-clip ==========
[cmd] D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe scripts/ood_multiclip_predict.py --checkpoint D:\VsCode Program\Python\content_security\final_project\runs\ood_stf3_new_224_fakew12\best.pt --batch-size 1 --clip-count 5 --clip-crop-ratio 0.6 --aggregate topk_mean --topk 2 --amp --csv data/GenVideo-Val/splits/ood_test.csv --out-dir D:\VsCode Program\Python\content_security\final_project\outputs\ood_followup\optimization_runs\r7_224_fakew12_5clip_top2_mean
[log] logs/ood_followup/r7_224_fakew12_5clip_top2_mean_test_multiclip.log


```text
R7_224_FAKEW12_5CLIP_TOP2_MEAN test multi-clip done, elapsed=39.0 min
predict: 100%|██████████| 3598/3598 [38:52<00:00,  1.54it/s]
```

Using cache found in C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
D:\VsCode Program\Python\content_security\final_project\src\models\fusion\branch_token_transformer.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
{'acc': 0.9238465814341301, 'auc': 0.97

```text
R7_224_FAKEW12_5CLIP_TOP2_MEAN val multi-clip done, elapsed=29.1 min
predict: 100%|██████████| 2429/2429 [28:52<00:00,  1.40it/s]
```

Using cache found in C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\jacky/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
D:\VsCode Program\Python\content_security\final_project\src\models\fusion\branch_token_transformer.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
{'acc': 0.9757101687937423, 'auc': 0.99

### 10.5 Multi-clip 统一阈值校准与错误分析

完成 10.3 或 10.4 后运行这一格。缺少 predictions 的 multi-clip 实验会自动跳过。

建议先看 `precision_0.95_recall_max`，再看 `max_acc`。


In [21]:
MULTICLIP_IDS = ["R7_224_FAKEW12_5CLIP_MEAN", "R7_224_FAKEW12_5CLIP_TOP2_MEAN"]

run_cmd([
    PYTHON, "scripts/ood_threshold_calibration.py",
    "--ids", ",".join(MULTICLIP_IDS),
    "--extra-experiments-json", rel(OPTIMIZATION_JSON),
    "--out-dir", "outputs/ood_followup/optimization_runs",
], "multi-clip threshold calibration", log_name="multiclip_threshold_calibration.log", force=True)

show_csv("outputs/ood_followup/optimization_runs/val_calibrated_thresholds.csv", n=80)

for obj in ["precision_0.95_recall_max", "max_acc"]:
    run_cmd([
        PYTHON, "scripts/ood_generator_error_analysis.py",
        "--ids", ",".join(MULTICLIP_IDS),
        "--extra-experiments-json", rel(OPTIMIZATION_JSON),
        "--thresholds-csv", "outputs/ood_followup/optimization_runs/val_calibrated_thresholds.csv",
        "--objective", obj,
        "--out-dir", "outputs/ood_followup/optimization_runs",
    ], f"multi-clip generator analysis {obj}", log_name=f"multiclip_generator_analysis_{obj}.log", force=True)
    show_csv(f"outputs/ood_followup/optimization_runs/generator_error_analysis_{obj}.csv", n=80)



========== multi-clip threshold calibration ==========
[cmd] D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe scripts/ood_threshold_calibration.py --ids R7_224_FAKEW12_5CLIP_MEAN,R7_224_FAKEW12_5CLIP_TOP2_MEAN --extra-experiments-json outputs/ood_followup/optimization_experiments.json --out-dir outputs/ood_followup/optimization_runs
[log] logs/ood_followup/multiclip_threshold_calibration.log


```text
multi-clip threshold calibration done, elapsed=0.2 min
completed
```

[write] outputs\ood_followup\optimization_runs\val_calibrated_thresholds.csv
[write] outputs\ood_followup\optimization_runs\val_calibrated_thresholds.md
[write] outputs\ood_followup\optimization_runs\val_calibrated_thresholds.json
[done] multi-clip threshold calibration, elapsed=0.2 min


,id,method,objective,threshold,val_acc,val_f1,val_precision,val_recall,val_balanced_acc,val_fn,...,test_recall,test_balanced_acc,test_fn,test_fp,default_test_acc,default_test_f1,default_test_precision,default_test_recall,default_test_fn,default_test_fp
0,R7_224_FAKEW12_5CLIP_MEAN,stf3_new_224_fakew12_5clip_mean,max_f1,0.769043,0.989296,0.985961,0.989166,0.982777,0.988055,16,...,0.852240,0.922120,310,12,0.925236,0.932834,0.979549,0.890372,230,39
1,R7_224_FAKEW12_5CLIP_MEAN,stf3_new_224_fakew12_5clip_mean,max_balanced_acc,0.769043,0.989296,0.985961,0.989166,0.982777,0.988055,16,...,0.852240,0.922120,310,12,0.925236,0.932834,0.979549,0.890372,230,39
2,R7_224_FAKEW12_5CLIP_MEAN,stf3_new_224_fakew12_5clip_mean,precision_0.95_recall_max,0.422607,0.983121,0.978249,0.964435,0.992465,0.984899,7,...,0.896568,0.931617,217,50,0.925236,0.932834,0.979549,0.890372,230,39
3,R7_224_FAKEW12_5CLIP_MEAN,stf3_new_224_fakew12_5clip_mean,max_acc,0.769043,0.989296,0.985961,0.989166,0.982777,0.988055,16,...,0.852240,0.922120,310,12,0.925236,0.932834,0.979549,0.890372,230,39
4,R7_224_FAKEW12_5CLIP_TOP2_MEAN,stf3_new_224_fakew12_5clip_top2_mean,max_f1,0.876221,0.985179,0.980624,0.980624,0.980624,0.984312,18,...,0.842707,0.911687,330,29,0.923847,0.932645,0.962944,0.904194,201,73
5,R7_224_FAKEW12_5CLIP_TOP2_MEAN,stf3_new_224_fakew12_5clip_top2_mean,max_balanced_acc,0.788574,0.983944,0.979156,0.972399,0.986006,0.984337,13,...,0.863680,0.919507,286,37,0.923847,0.932645,0.962944,0.904194,201,73
6,R7_224_FAKEW12_5CLIP_TOP2_MEAN,stf3_new_224_fakew12_5clip_top2_mean,precision_0.95_recall_max,0.562012,0.978592,0.972487,0.956296,0.989236,0.980618,10,...,0.897521,0.927761,215,63,0.923847,0.932645,0.962944,0.904194,201,73
7,R7_224_FAKEW12_5CLIP_TOP2_MEAN,stf3_new_224_fakew12_5clip_top2_mean,max_acc,0.876221,0.985179,0.980624,0.980624,0.980624,0.984312,18,...,0.842707,0.911687,330,29,0.923847,0.932645,0.962944,0.904194,201,73


rows=8 path=outputs/ood_followup/optimization_runs/val_calibrated_thresholds.csv

========== multi-clip generator analysis precision_0.95_recall_max ==========
[cmd] D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe scripts/ood_generator_error_analysis.py --ids R7_224_FAKEW12_5CLIP_MEAN,R7_224_FAKEW12_5CLIP_TOP2_MEAN --extra-experiments-json outputs/ood_followup/optimization_experiments.json --thresholds-csv outputs/ood_followup/optimization_runs/val_calibrated_thresholds.csv --objective precision_0.95_recall_max --out-dir outputs/ood_followup/optimization_runs
[log] logs/ood_followup/multiclip_generator_analysis_precision_0.95_recall_max.log


...

rows=10 path=outputs/ood_followup/optimization_runs/generator_error_analysis_precision_0.95_recall_max.csv

========== multi-clip generator analysis max_acc ==========
[cmd] D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe scripts/ood_generator_error_analysis.py --ids R7_224_FAKEW12_5CLIP_MEAN,R7_224_FAKEW12_5CLIP_TOP2_MEAN --extra-experiments-json outputs/ood_followup/optimization_experiments.json --thresholds-csv outputs/ood_followup/optimization_runs/val_calibrated_thresholds.csv --objective max_acc --out-dir outputs/ood_followup/optimization_runs
[log] logs/ood_followup/multiclip_generator_analysis_max_acc.log


```text
multi-clip generator analysis max_acc done, elapsed=0.0 min
completed
```

[write] outputs\ood_followup\optimization_runs\generator_error_analysis_max_acc.csv
[write] outputs\ood_followup\optimization_runs\r5_r7_fn_comparison_max_acc.csv
[write] outputs\ood_followup\optimization_runs\generator_error_analysis_max_acc.md
[done] multi-clip generator analysis max_acc, elapsed=0.0 min


,generator,n,real_n,fake_n,acc,precision,recall,specificity,tn,fp,fn,tp,id,method,threshold
0,MorphStudio,700,0,700,0.964286,1.0,0.964286,NaN,0,0,25,675,R7_224_FAKEW12_5CLIP_MEAN,stf3_new_224_fakew12_5clip_mean,0.769043
1,Show_1,700,0,700,0.941429,1.0,0.941429,NaN,0,0,41,659,R7_224_FAKEW12_5CLIP_MEAN,stf3_new_224_fakew12_5clip_mean,0.769043
2,Sora,56,0,56,0.857143,1.0,0.857143,NaN,0,0,8,48,R7_224_FAKEW12_5CLIP_MEAN,stf3_new_224_fakew12_5clip_mean,0.769043
3,WildScrape,642,0,642,0.632399,1.0,0.632399,NaN,0,0,236,406,R7_224_FAKEW12_5CLIP_MEAN,stf3_new_224_fakew12_5clip_mean,0.769043
4,real_MSRVTT,1500,1500,0,0.992000,0.0,NaN,0.992000,1488,12,0,0,R7_224_FAKEW12_5CLIP_MEAN,stf3_new_224_fakew12_5clip_mean,0.769043
5,MorphStudio,700,0,700,0.955714,1.0,0.955714,NaN,0,0,31,669,R7_224_FAKEW12_5CLIP_TOP2_MEAN,stf3_new_224_fakew12_5clip_top2_mean,0.876221
6,Show_1,700,0,700,0.935714,1.0,0.935714,NaN,0,0,45,655,R7_224_FAKEW12_5CLIP_TOP2_MEAN,stf3_new_224_fakew12_5clip_top2_mean,0.876221
7,Sora,56,0,56,0.875000,1.0,0.875000,NaN,0,0,7,49,R7_224_FAKEW12_5CLIP_TOP2_MEAN,stf3_new_224_fakew12_5clip_top2_mean,0.876221
8,WildScrape,642,0,642,0.615265,1.0,0.615265,NaN,0,0,247,395,R7_224_FAKEW12_5CLIP_TOP2_MEAN,stf3_new_224_fakew12_5clip_top2_mean,0.876221
9,real_MSRVTT,1500,1500,0,0.980667,0.0,NaN,0.980667,1471,29,0,0,R7_224_FAKEW12_5CLIP_TOP2_MEAN,stf3_new_224_fakew12_5clip_top2_mean,0.876221


rows=10 path=outputs/ood_followup/optimization_runs/generator_error_analysis_max_acc.csv


### 10.6 STF3-only 概率融合：`R7_224_FAKEW12 + R7_224`

这个实验不引入 R5，只融合两个 STF3/R7 变体的概率：

```text
p = w * p_R7_224_FAKEW12 + (1 - w) * p_R7_224
```

权重和阈值都在 val 上搜索，然后固定到 test。


In [10]:
run_cmd([
    PYTHON, "scripts/ood_stf3_variant_ensemble.py",
    "--a-id", "R7_224_FAKEW12",
    "--a-method", "stf3_new_224_fakew12",
    "--a-out-dir", "outputs/ood_stf3_new_224_fakew12",
    "--b-id", "R7_224",
    "--b-method", "stf3_new_224",
    "--b-out-dir", "outputs/ood_stf3_new_224",
    "--out-dir", "outputs/ood_followup/optimization_runs",
    "--prefix", "r7_224_fakew12_plus_r7_224",
], "STF3-only ensemble R7_224_FAKEW12 + R7_224", log_name="stf3_only_ensemble_r7_224_fakew12_plus_r7_224.log", force=True)

show_csv("outputs/ood_followup/optimization_runs/r7_224_fakew12_plus_r7_224_ensemble_results.csv", n=20)



========== STF3-only ensemble R7_224_FAKEW12 + R7_224 ==========
[cmd] D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe scripts/ood_stf3_variant_ensemble.py --a-id R7_224_FAKEW12 --a-method stf3_new_224_fakew12 --a-out-dir outputs/ood_stf3_new_224_fakew12 --b-id R7_224 --b-method stf3_new_224 --b-out-dir outputs/ood_stf3_new_224 --out-dir outputs/ood_followup/optimization_runs --prefix r7_224_fakew12_plus_r7_224
[log] logs/ood_followup/stf3_only_ensemble_r7_224_fakew12_plus_r7_224.log


```text
STF3-only ensemble R7_224_FAKEW12 + R7_224 done, elapsed=0.1 min
completed
```

[write] outputs\ood_followup\optimization_runs\r7_224_fakew12_plus_r7_224_ensemble_results.csv
[write] outputs\ood_followup\optimization_runs\r7_224_fakew12_plus_r7_224_ensemble_results.md
[write] outputs\ood_followup\optimization_runs\r7_224_fakew12_plus_r7_224_ensemble_results.json
[done] STF3-only ensemble R7_224_FAKEW12 + R7_224, elapsed=0.1 min


,objective,a_id,a_method,b_id,b_method,weight_a,weight_b,threshold,val_acc,val_f1,...,val_fp,test_acc,test_f1,test_precision,test_recall,test_balanced_acc,test_tn,test_fn,test_fp,test_tp
0,max_f1,R7_224_FAKEW12,stf3_new_224_fakew12,R7_224,stf3_new_224,0.60,0.40,0.288086,0.990943,0.988223,...,16,0.931907,0.938643,0.988918,0.893232,0.939616,1479,224,21,1874
1,max_balanced_acc,R7_224_FAKEW12,stf3_new_224_fakew12,R7_224,stf3_new_224,0.60,0.40,0.288086,0.990943,0.988223,...,16,0.931907,0.938643,0.988918,0.893232,0.939616,1479,224,21,1874
2,precision_0.95_recall_max,R7_224_FAKEW12,stf3_new_224_fakew12,R7_224,stf3_new_224,0.55,0.45,0.031300,0.980650,0.975302,...,46,0.943858,0.950756,0.973054,0.929457,0.946728,1446,148,54,1950
3,max_acc,R7_224_FAKEW12,stf3_new_224_fakew12,R7_224,stf3_new_224,0.60,0.40,0.288086,0.990943,0.988223,...,16,0.931907,0.938643,0.988918,0.893232,0.939616,1479,224,21,1874


rows=4 path=outputs/ood_followup/optimization_runs/r7_224_fakew12_plus_r7_224_ensemble_results.csv


,objective,a_id,a_method,b_id,b_method,weight_a,weight_b,threshold,val_acc,val_f1,...,val_fp,test_acc,test_f1,test_precision,test_recall,test_balanced_acc,test_tn,test_fn,test_fp,test_tp
0,max_f1,R7_224_FAKEW12,stf3_new_224_fakew12,R7_224,stf3_new_224,0.60,0.40,0.288086,0.990943,0.988223,...,16,0.931907,0.938643,0.988918,0.893232,0.939616,1479,224,21,1874
1,max_balanced_acc,R7_224_FAKEW12,stf3_new_224_fakew12,R7_224,stf3_new_224,0.60,0.40,0.288086,0.990943,0.988223,...,16,0.931907,0.938643,0.988918,0.893232,0.939616,1479,224,21,1874
2,precision_0.95_recall_max,R7_224_FAKEW12,stf3_new_224_fakew12,R7_224,stf3_new_224,0.55,0.45,0.031300,0.980650,0.975302,...,46,0.943858,0.950756,0.973054,0.929457,0.946728,1446,148,54,1950
3,max_acc,R7_224_FAKEW12,stf3_new_224_fakew12,R7_224,stf3_new_224,0.60,0.40,0.288086,0.990943,0.988223,...,16,0.931907,0.938643,0.988918,0.893232,0.939616,1479,224,21,1874


### 10.7 STF3 分支级 LR 校准

这个实验只用 `R7_224_FAKEW12` 自己的输出和分支信息，在 val 上拟合 logistic regression 校准器，然后固定到 test。

使用特征包括：

```text
fake_prob
fake_prob_spatial
fake_prob_temporal
fake_prob_frequency
branch_weight_spatial
branch_weight_temporal
branch_weight_frequency
```


In [11]:
run_cmd([
    PYTHON, "scripts/ood_branch_calibration_lr.py",
    "--id", "R7_224_FAKEW12",
    "--method", "stf3_new_224_fakew12_branch_lr",
    "--out-dir-input", "outputs/ood_stf3_new_224_fakew12",
    "--out-dir", "outputs/ood_followup/optimization_runs",
    "--prefix", "r7_224_fakew12_branch_lr",
], "R7_224_FAKEW12 branch calibration LR", log_name="r7_224_fakew12_branch_calibration_lr.log", force=True)

show_csv("outputs/ood_followup/optimization_runs/r7_224_fakew12_branch_lr_results.csv", n=20)



========== R7_224_FAKEW12 branch calibration LR ==========
[cmd] D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe scripts/ood_branch_calibration_lr.py --id R7_224_FAKEW12 --method stf3_new_224_fakew12_branch_lr --out-dir-input outputs/ood_stf3_new_224_fakew12 --out-dir outputs/ood_followup/optimization_runs --prefix r7_224_fakew12_branch_lr
[log] logs/ood_followup/r7_224_fakew12_branch_calibration_lr.log


```text
R7_224_FAKEW12 branch calibration LR done, elapsed=0.4 min
completed
```

[write] outputs\ood_followup\optimization_runs\r7_224_fakew12_branch_lr_results.csv
[write] outputs\ood_followup\optimization_runs\r7_224_fakew12_branch_lr_results.md
[write] outputs\ood_followup\optimization_runs\r7_224_fakew12_branch_lr_results.json
[done] R7_224_FAKEW12 branch calibration LR, elapsed=0.4 min


,id,method,objective,features,threshold,val_acc,val_f1,val_precision,val_recall,val_balanced_acc,...,test_f1,test_precision,test_recall,test_balanced_acc,test_auc,test_ap,test_tn,test_fn,test_fp,test_tp
0,R7_224_FAKEW12,stf3_new_224_fakew12_branch_lr,max_f1,"fake_prob,fake_prob_spatial,fake_prob_temporal...",0.523553,0.991766,0.989270,0.986096,0.992465,0.991899,...,0.940231,0.993631,0.892278,0.942139,0.989229,0.993611,1488,226,12,1872
1,R7_224_FAKEW12,stf3_new_224_fakew12_branch_lr,max_balanced_acc,"fake_prob,fake_prob_spatial,fake_prob_temporal...",0.322710,0.991354,0.988764,0.982979,0.994618,0.991976,...,0.942907,0.988500,0.901335,0.943334,0.989229,0.993611,1478,207,22,1891
2,R7_224_FAKEW12,stf3_new_224_fakew12_branch_lr,precision_0.95_recall_max,"fake_prob,fake_prob_spatial,fake_prob_temporal...",0.099308,0.986414,0.982512,0.967641,0.997847,0.988590,...,0.953010,0.979376,0.928027,0.950347,0.989229,0.993611,1459,151,41,1947
3,R7_224_FAKEW12,stf3_new_224_fakew12_branch_lr,max_acc,"fake_prob,fake_prob_spatial,fake_prob_temporal...",0.523553,0.991766,0.989270,0.986096,0.992465,0.991899,...,0.940231,0.993631,0.892278,0.942139,0.989229,0.993611,1488,226,12,1872


rows=4 path=outputs/ood_followup/optimization_runs/r7_224_fakew12_branch_lr_results.csv


,id,method,objective,features,threshold,val_acc,val_f1,val_precision,val_recall,val_balanced_acc,...,test_f1,test_precision,test_recall,test_balanced_acc,test_auc,test_ap,test_tn,test_fn,test_fp,test_tp
0,R7_224_FAKEW12,stf3_new_224_fakew12_branch_lr,max_f1,"fake_prob,fake_prob_spatial,fake_prob_temporal...",0.523553,0.991766,0.989270,0.986096,0.992465,0.991899,...,0.940231,0.993631,0.892278,0.942139,0.989229,0.993611,1488,226,12,1872
1,R7_224_FAKEW12,stf3_new_224_fakew12_branch_lr,max_balanced_acc,"fake_prob,fake_prob_spatial,fake_prob_temporal...",0.322710,0.991354,0.988764,0.982979,0.994618,0.991976,...,0.942907,0.988500,0.901335,0.943334,0.989229,0.993611,1478,207,22,1891
2,R7_224_FAKEW12,stf3_new_224_fakew12_branch_lr,precision_0.95_recall_max,"fake_prob,fake_prob_spatial,fake_prob_temporal...",0.099308,0.986414,0.982512,0.967641,0.997847,0.988590,...,0.953010,0.979376,0.928027,0.950347,0.989229,0.993611,1459,151,41,1947
3,R7_224_FAKEW12,stf3_new_224_fakew12_branch_lr,max_acc,"fake_prob,fake_prob_spatial,fake_prob_temporal...",0.523553,0.991766,0.989270,0.986096,0.992465,0.991899,...,0.940231,0.993631,0.892278,0.942139,0.989229,0.993611,1488,226,12,1872


### 10.8 本节结果查看

运行完任意优化实验后，可以用这一格快速查看关键结果文件。


In [22]:
result_files = [
    "outputs/ood_followup/followup_runs/val_calibrated_thresholds.csv",
    "outputs/ood_followup/optimization_runs/val_calibrated_thresholds.csv",
    "outputs/ood_followup/optimization_runs/r7_224_fakew12_plus_r7_224_ensemble_results.csv",
    "outputs/ood_followup/optimization_runs/r7_224_fakew12_branch_lr_results.csv",
]

for file in result_files:
    print("==", file, "==")
    show_csv(file, n=20)


== outputs/ood_followup/followup_runs/val_calibrated_thresholds.csv ==


,id,method,objective,threshold,val_acc,val_f1,val_precision,val_recall,val_balanced_acc,val_fn,...,test_recall,test_balanced_acc,test_fn,test_fp,default_test_acc,default_test_f1,default_test_precision,default_test_recall,default_test_fn,default_test_fp
0,R7_E6,stf3_new_e6,max_f1,0.112793,0.980239,0.974304,0.969116,0.979548,0.980107,19,...,0.867016,0.927508,279,18,0.896609,0.903226,0.994273,0.827455,362,10
1,R7_E6,stf3_new_e6,max_balanced_acc,0.112793,0.980239,0.974304,0.969116,0.979548,0.980107,19,...,0.867016,0.927508,279,18,0.896609,0.903226,0.994273,0.827455,362,10
2,R7_E6,stf3_new_e6,precision_0.95_recall_max,0.017303,0.976534,0.969953,0.950413,0.990312,0.979156,9,...,0.895138,0.934902,220,38,0.896609,0.903226,0.994273,0.827455,362,10
3,R7_E6,stf3_new_e6,max_acc,0.112793,0.980239,0.974304,0.969116,0.979548,0.980107,19,...,0.867016,0.927508,279,18,0.896609,0.903226,0.994273,0.827455,362,10
4,R7_FAKEW12,stf3_new_fakew12,max_f1,0.569336,0.979827,0.973499,0.978261,0.968784,0.977725,29,...,0.857007,0.922837,300,17,0.916342,0.923468,0.989646,0.865586,282,19
5,R7_FAKEW12,stf3_new_fakew12,max_balanced_acc,0.315186,0.979415,0.973433,0.961175,0.986006,0.980670,13,...,0.887512,0.934089,236,29,0.916342,0.923468,0.989646,0.865586,282,19
6,R7_FAKEW12,stf3_new_fakew12,precision_0.95_recall_max,0.245728,0.977357,0.971007,0.951446,0.991389,0.980028,8,...,0.894662,0.935997,221,34,0.916342,0.923468,0.989646,0.865586,282,19
7,R7_FAKEW12,stf3_new_fakew12,max_acc,0.569336,0.979827,0.973499,0.978261,0.968784,0.977725,29,...,0.857007,0.922837,300,17,0.916342,0.923468,0.989646,0.865586,282,19
8,R7_224,stf3_new_224,max_f1,0.524902,0.988884,0.985398,0.990217,0.980624,0.987312,18,...,0.837464,0.916399,341,7,0.904391,0.911019,0.996041,0.839371,337,7
9,R7_224,stf3_new_224,max_balanced_acc,0.056122,0.988061,0.984467,0.979744,0.989236,0.988285,10,...,0.878932,0.933466,254,18,0.904391,0.911019,0.996041,0.839371,337,7


rows=16 path=outputs/ood_followup/followup_runs/val_calibrated_thresholds.csv
== outputs/ood_followup/optimization_runs/val_calibrated_thresholds.csv ==


,id,method,objective,threshold,val_acc,val_f1,val_precision,val_recall,val_balanced_acc,val_fn,...,test_recall,test_balanced_acc,test_fn,test_fp,default_test_acc,default_test_f1,default_test_precision,default_test_recall,default_test_fn,default_test_fp
0,R7_224_FAKEW12_5CLIP_MEAN,stf3_new_224_fakew12_5clip_mean,max_f1,0.769043,0.989296,0.985961,0.989166,0.982777,0.988055,16,...,0.852240,0.922120,310,12,0.925236,0.932834,0.979549,0.890372,230,39
1,R7_224_FAKEW12_5CLIP_MEAN,stf3_new_224_fakew12_5clip_mean,max_balanced_acc,0.769043,0.989296,0.985961,0.989166,0.982777,0.988055,16,...,0.852240,0.922120,310,12,0.925236,0.932834,0.979549,0.890372,230,39
2,R7_224_FAKEW12_5CLIP_MEAN,stf3_new_224_fakew12_5clip_mean,precision_0.95_recall_max,0.422607,0.983121,0.978249,0.964435,0.992465,0.984899,7,...,0.896568,0.931617,217,50,0.925236,0.932834,0.979549,0.890372,230,39
3,R7_224_FAKEW12_5CLIP_MEAN,stf3_new_224_fakew12_5clip_mean,max_acc,0.769043,0.989296,0.985961,0.989166,0.982777,0.988055,16,...,0.852240,0.922120,310,12,0.925236,0.932834,0.979549,0.890372,230,39
4,R7_224_FAKEW12_5CLIP_TOP2_MEAN,stf3_new_224_fakew12_5clip_top2_mean,max_f1,0.876221,0.985179,0.980624,0.980624,0.980624,0.984312,18,...,0.842707,0.911687,330,29,0.923847,0.932645,0.962944,0.904194,201,73
5,R7_224_FAKEW12_5CLIP_TOP2_MEAN,stf3_new_224_fakew12_5clip_top2_mean,max_balanced_acc,0.788574,0.983944,0.979156,0.972399,0.986006,0.984337,13,...,0.863680,0.919507,286,37,0.923847,0.932645,0.962944,0.904194,201,73
6,R7_224_FAKEW12_5CLIP_TOP2_MEAN,stf3_new_224_fakew12_5clip_top2_mean,precision_0.95_recall_max,0.562012,0.978592,0.972487,0.956296,0.989236,0.980618,10,...,0.897521,0.927761,215,63,0.923847,0.932645,0.962944,0.904194,201,73
7,R7_224_FAKEW12_5CLIP_TOP2_MEAN,stf3_new_224_fakew12_5clip_top2_mean,max_acc,0.876221,0.985179,0.980624,0.980624,0.980624,0.984312,18,...,0.842707,0.911687,330,29,0.923847,0.932645,0.962944,0.904194,201,73


rows=8 path=outputs/ood_followup/optimization_runs/val_calibrated_thresholds.csv
== outputs/ood_followup/optimization_runs/r7_224_fakew12_plus_r7_224_ensemble_results.csv ==


,objective,a_id,a_method,b_id,b_method,weight_a,weight_b,threshold,val_acc,val_f1,...,val_fp,test_acc,test_f1,test_precision,test_recall,test_balanced_acc,test_tn,test_fn,test_fp,test_tp
0,max_f1,R7_224_FAKEW12,stf3_new_224_fakew12,R7_224,stf3_new_224,0.60,0.40,0.288086,0.990943,0.988223,...,16,0.931907,0.938643,0.988918,0.893232,0.939616,1479,224,21,1874
1,max_balanced_acc,R7_224_FAKEW12,stf3_new_224_fakew12,R7_224,stf3_new_224,0.60,0.40,0.288086,0.990943,0.988223,...,16,0.931907,0.938643,0.988918,0.893232,0.939616,1479,224,21,1874
2,precision_0.95_recall_max,R7_224_FAKEW12,stf3_new_224_fakew12,R7_224,stf3_new_224,0.55,0.45,0.031300,0.980650,0.975302,...,46,0.943858,0.950756,0.973054,0.929457,0.946728,1446,148,54,1950
3,max_acc,R7_224_FAKEW12,stf3_new_224_fakew12,R7_224,stf3_new_224,0.60,0.40,0.288086,0.990943,0.988223,...,16,0.931907,0.938643,0.988918,0.893232,0.939616,1479,224,21,1874


rows=4 path=outputs/ood_followup/optimization_runs/r7_224_fakew12_plus_r7_224_ensemble_results.csv
== outputs/ood_followup/optimization_runs/r7_224_fakew12_branch_lr_results.csv ==


,id,method,objective,features,threshold,val_acc,val_f1,val_precision,val_recall,val_balanced_acc,...,test_f1,test_precision,test_recall,test_balanced_acc,test_auc,test_ap,test_tn,test_fn,test_fp,test_tp
0,R7_224_FAKEW12,stf3_new_224_fakew12_branch_lr,max_f1,"fake_prob,fake_prob_spatial,fake_prob_temporal...",0.523553,0.991766,0.989270,0.986096,0.992465,0.991899,...,0.940231,0.993631,0.892278,0.942139,0.989229,0.993611,1488,226,12,1872
1,R7_224_FAKEW12,stf3_new_224_fakew12_branch_lr,max_balanced_acc,"fake_prob,fake_prob_spatial,fake_prob_temporal...",0.322710,0.991354,0.988764,0.982979,0.994618,0.991976,...,0.942907,0.988500,0.901335,0.943334,0.989229,0.993611,1478,207,22,1891
2,R7_224_FAKEW12,stf3_new_224_fakew12_branch_lr,precision_0.95_recall_max,"fake_prob,fake_prob_spatial,fake_prob_temporal...",0.099308,0.986414,0.982512,0.967641,0.997847,0.988590,...,0.953010,0.979376,0.928027,0.950347,0.989229,0.993611,1459,151,41,1947
3,R7_224_FAKEW12,stf3_new_224_fakew12_branch_lr,max_acc,"fake_prob,fake_prob_spatial,fake_prob_temporal...",0.523553,0.991766,0.989270,0.986096,0.992465,0.991899,...,0.940231,0.993631,0.892278,0.942139,0.989229,0.993611,1488,226,12,1872


rows=4 path=outputs/ood_followup/optimization_runs/r7_224_fakew12_branch_lr_results.csv


## 11. 保持 STF3 独立性的结构不变优化实验

本节只优化 STF3 自身训练与输入采样，不加入外部检测器、独立 D3/ReStraV 分数或额外后处理模型。

- **11.1 有效 WaveRep donor bank**：保持三分支与融合结构不变，让 batch size = 1 时小波频带替换真正生效。
- **11.2 完整视频 16 帧采样**：保持三分支与融合结构不变，仅将均匀采样帧数从 8 增加到 16。
- 两个实验都以当前最佳单模型配置 `R7_224_FAKEW12` 为基准：`stf3_new + 224 + fake_loss_weight=1.2 + 5 epochs`。
- 一次只运行一个训练 cell。首次运行前确保第 1、2、3 节的环境、基础配置和命令函数 cell 已运行，并且 `RUN_LONG_TRAINING = True`。


In [6]:
INDEPENDENT_STF3_EXPERIMENTS = {
    "R7_224_FAKEW12_WAVEREP_BANK": {
        "method": "stf3_new_224_fakew12_waverep_bank",
        "model": "stf3_new",
        "run_dir": "runs/ood_stf3_new_224_fakew12_waverep_bank",
        "out_dir": "outputs/ood_stf3_new_224_fakew12_waverep_bank",
        "epochs": 5, "image_size": 224, "num_frames": 8,
        "fake_loss_weight": 1.2, "pos_weight": "none",
        "wavelet_aug_prob": 0.1, "wavelet_aug_mode": "bank",
    },
    "R7_224_F16_FAKEW12": {
        "method": "stf3_new_224_f16_fakew12",
        "model": "stf3_new",
        "run_dir": "runs/ood_stf3_new_224_f16_fakew12",
        "out_dir": "outputs/ood_stf3_new_224_f16_fakew12",
        "epochs": 5, "image_size": 224, "num_frames": 16,
        "fake_loss_weight": 1.2, "pos_weight": "none",
        "wavelet_aug_prob": 0.0, "wavelet_aug_mode": "batch",
    },
}

INDEPENDENT_STF3_JSON = OUT_DIR / "independent_stf3_experiments.json"
INDEPENDENT_STF3_ANALYSIS_EXPERIMENTS = {
    "R7_224_FAKEW12": {"method": "stf3_new_224_fakew12", "out_dir": "outputs/ood_stf3_new_224_fakew12"},
    **{k: {"method": v["method"], "out_dir": v["out_dir"]} for k, v in INDEPENDENT_STF3_EXPERIMENTS.items()},
}
INDEPENDENT_STF3_JSON.write_text(
    json.dumps(INDEPENDENT_STF3_ANALYSIS_EXPERIMENTS, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

def train_eval_independent_stf3(exp_id, force=False):
    cfg = INDEPENDENT_STF3_EXPERIMENTS[exp_id]
    run_dir = PROJECT_ROOT / cfg["run_dir"]
    out_dir = PROJECT_ROOT / cfg["out_dir"]
    val_out_dir = PROJECT_ROOT / f"{cfg['out_dir']}_val"
    if not RUN_LONG_TRAINING:
        print(f"[guard] RUN_LONG_TRAINING=False, skip {exp_id}. Set it to True when ready.")
        return

    train_cmd = [
        PYTHON, "-m", "src.train",
        "--model", cfg["model"],
        "--epochs", str(cfg["epochs"]),
        "--image-size", str(cfg["image_size"]),
        "--num-frames", str(cfg["num_frames"]),
        "--fake-loss-weight", str(cfg["fake_loss_weight"]),
        "--pos-weight", cfg["pos_weight"],
        "--wavelet-aug-prob", str(cfg["wavelet_aug_prob"]),
        "--wavelet-aug-mode", cfg["wavelet_aug_mode"],
        "--train-csv", TRAIN_CSV,
        "--val-csv", VAL_CSV,
        "--batch-size", "1",
        "--foundation-backbone", "dinov2_vits14",
        "--branch-dropout", "0.1",
        "--aux-loss-weight", "0.2",
        "--amp",
        "--out-dir", run_dir,
    ]
    run_cmd(train_cmd, f"{exp_id} train", log_name=f"{exp_id.lower()}_train.log", skip_if=run_dir / "best.pt", force=force)

    for split_name, split_csv, split_out in [("test", TEST_CSV, out_dir), ("val", VAL_CSV, val_out_dir)]:
        run_cmd([
            PYTHON, "-m", "src.evaluate",
            "--checkpoint", run_dir / "best.pt",
            "--csv", split_csv,
            "--batch-size", "1",
            "--amp",
            "--out-dir", split_out,
        ], f"{exp_id} {split_name} evaluate", log_name=f"{exp_id.lower()}_{split_name}_evaluate.log", skip_if=split_out / "predictions.csv", force=force or FORCE_EVAL)

    print(f"[ready] {exp_id}: 完成后运行 11.3 做统一阈值校准与 generator 分析。")

print("[write]", rel(INDEPENDENT_STF3_JSON))
display(pd.DataFrame([{"ID": k, **v} for k, v in INDEPENDENT_STF3_EXPERIMENTS.items()]))


[write] outputs/ood_followup/independent_stf3_experiments.json


,ID,method,model,run_dir,out_dir,epochs,image_size,num_frames,fake_loss_weight,pos_weight,wavelet_aug_prob,wavelet_aug_mode
0,R7_224_FAKEW12_WAVEREP_BANK,stf3_new_224_fakew12_waverep_bank,stf3_new,runs/ood_stf3_new_224_fakew12_waverep_bank,outputs/ood_stf3_new_224_fakew12_waverep_bank,5,224,8,1.2,none,0.1,bank
1,R7_224_F16_FAKEW12,stf3_new_224_f16_fakew12,stf3_new,runs/ood_stf3_new_224_f16_fakew12,outputs/ood_stf3_new_224_f16_fakew12,5,224,16,1.2,none,0.0,batch


### 11.1 有效 WaveRep donor-bank 增强

该实验仍是原始三分支 STF3。变化仅发生在训练输入增强：上一条训练视频作为当前视频的小波频带 donor，使 `batch_size=1` 时增强能够生效。推理阶段不使用 donor bank。


In [8]:
train_eval_independent_stf3("R7_224_FAKEW12_WAVEREP_BANK")


[skip] R7_224_FAKEW12_WAVEREP_BANK train: runs/ood_stf3_new_224_fakew12_waverep_bank/best.pt already exists

========== R7_224_FAKEW12_WAVEREP_BANK test evaluate ==========
[cmd] D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe -m src.evaluate --checkpoint D:\VsCode Program\Python\content_security\final_project\runs\ood_stf3_new_224_fakew12_waverep_bank\best.pt --csv data/GenVideo-Val/splits/ood_test.csv --batch-size 1 --amp --out-dir D:\VsCode Program\Python\content_security\final_project\outputs\ood_stf3_new_224_fakew12_waverep_bank
[log] logs/ood_followup/r7_224_fakew12_waverep_bank_test_evaluate.log


```text
R7_224_FAKEW12_WAVEREP_BANK test evaluate done, elapsed=11.4 min
predict: 100%|██████████| 3598/3598 [11:16<00:00,  5.32it/s]
```

C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
D:\VsCode Program\Python\content_security\final_project\src\models\fusion\branch_token_transformer.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
{'acc': 0.9124513618677043, 'auc': 0.9733277724817287, 'f1': 0.9192100538599641, 'precision': 0.9950027762354248, 'recall

```text
R7_224_FAKEW12_WAVEREP_BANK val evaluate done, elapsed=7.4 min
predict: 100%|██████████| 2429/2429 [07:16<00:00,  5.56it/s]
```

C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
D:\VsCode Program\Python\content_security\final_project\src\models\fusion\branch_token_transformer.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
{'acc': 0.9884726224783862, 'auc': 0.9978263365626121, 'f1': 0.9848975188781014, 'precision': 0.987027027027027, 'recall'

### 11.2 完整视频均匀采样 16 帧

该实验仍是原始三分支 STF3。仅把完整视频范围内的均匀采样帧数从 8 改为 16；为了单独判断帧数作用，本实验关闭 WaveRep 增强。预计训练和评估耗时约为 8 帧实验的 1.7 至 2 倍。


In [7]:
train_eval_independent_stf3("R7_224_F16_FAKEW12")



========== R7_224_F16_FAKEW12 train ==========
[cmd] D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe -m src.train --model stf3_new --epochs 5 --image-size 224 --num-frames 16 --fake-loss-weight 1.2 --pos-weight none --wavelet-aug-prob 0.0 --wavelet-aug-mode batch --train-csv data/GenVideo-Val/splits/ood_train.csv --val-csv data/GenVideo-Val/splits/ood_val.csv --batch-size 1 --foundation-backbone dinov2_vits14 --branch-dropout 0.1 --aux-loss-weight 0.2 --amp --out-dir D:\VsCode Program\Python\content_security\final_project\runs\ood_stf3_new_224_f16_fakew12
[log] logs/ood_followup/r7_224_f16_fakew12_train.log


```text
R7_224_F16_FAKEW12 train done, elapsed=406.6 min
eval: 100%|██████████| 2429/2429 [12:54<00:00,  6.14it/s]
```

[device] cuda
[model] stf3_new
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
D:\VsCode Program\Python\content_security\final_project\src\models\fusion\branch_token_transformer.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
[loss] class_weight real=1.0000 fake=1.2000
[params] trainable=1,863,407 total=23,919,983


```text
R7_224_F16_FAKEW12 test evaluate done, elapsed=19.5 min
predict: 100%|██████████| 3598/3598 [19:04<00:00,  3.14it/s]
```

C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
D:\VsCode Program\Python\content_security\final_project\src\models\fusion\branch_token_transformer.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
{'acc': 0.9085603112840467, 'auc': 0.96384795042898, 'f1': 0.9154893398407398, 'precision': 0.992757660167131, 'recall': 

```text
R7_224_F16_FAKEW12 val evaluate done, elapsed=13.0 min
predict: 100%|██████████| 2429/2429 [12:52<00:00,  3.15it/s]
```

C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\jacky\.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
D:\VsCode Program\Python\content_security\final_project\src\models\fusion\branch_token_transformer.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
{'acc': 0.9872375463153561, 'auc': 0.9988148546824542, 'f1': 0.9832703723691312, 'precision': 0.9859307359307359, 'recall

### 11.3 两个实验完成后的统一校准与错误分析

只有对应新实验已经生成 val/test predictions 时，才把它保留在 `INDEPENDENT_STF3_IDS` 中。当前最佳基准 `R7_224_FAKEW12` 已包含在比较中。该 cell 会重新从验证集选择阈值，并输出总体指标与 generator 级错误分析。


In [8]:
INDEPENDENT_STF3_IDS = ["R7_224_FAKEW12", "R7_224_FAKEW12_WAVEREP_BANK", "R7_224_F16_FAKEW12"]
INDEPENDENT_ANALYSIS_DIR = "outputs/ood_followup/independent_stf3_runs"

run_cmd([
    PYTHON, "scripts/ood_threshold_calibration.py",
    "--ids", ",".join(INDEPENDENT_STF3_IDS),
    "--extra-experiments-json", rel(INDEPENDENT_STF3_JSON),
    "--out-dir", INDEPENDENT_ANALYSIS_DIR,
], "independent STF3 threshold calibration", log_name="independent_stf3_threshold_calibration.log", force=True)

run_cmd([
    PYTHON, "scripts/ood_generator_error_analysis.py",
    "--ids", ",".join(INDEPENDENT_STF3_IDS),
    "--extra-experiments-json", rel(INDEPENDENT_STF3_JSON),
    "--thresholds-csv", f"{INDEPENDENT_ANALYSIS_DIR}/val_calibrated_thresholds.csv",
    "--objective", "precision_0.95_recall_max",
    "--out-dir", INDEPENDENT_ANALYSIS_DIR,
], "independent STF3 generator analysis", log_name="independent_stf3_generator_analysis.log", force=True)

show_csv(f"{INDEPENDENT_ANALYSIS_DIR}/val_calibrated_thresholds.csv", n=30)
show_csv(f"{INDEPENDENT_ANALYSIS_DIR}/generator_error_analysis_precision_0.95_recall_max.csv", n=50)



========== independent STF3 threshold calibration ==========
[cmd] D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe scripts/ood_threshold_calibration.py --ids R7_224_FAKEW12,R7_224_FAKEW12_WAVEREP_BANK,R7_224_F16_FAKEW12 --extra-experiments-json outputs/ood_followup/independent_stf3_experiments.json --out-dir outputs/ood_followup/independent_stf3_runs
[log] logs/ood_followup/independent_stf3_threshold_calibration.log


```text
independent STF3 threshold calibration done, elapsed=0.2 min
completed
```

[write] outputs\ood_followup\independent_stf3_runs\val_calibrated_thresholds.csv
[write] outputs\ood_followup\independent_stf3_runs\val_calibrated_thresholds.md
[write] outputs\ood_followup\independent_stf3_runs\val_calibrated_thresholds.json
[done] independent STF3 threshold calibration, elapsed=0.2 min

========== independent STF3 generator analysis ==========
[cmd] D:\VsCode Program\Python\content_security\final_project\.venv\Scripts\python.exe scripts/ood_generator_error_analysis.py --ids R7_224_FAKEW12,R7_224_FAKEW12_WAVEREP_BANK,R7_224_F16_FAKEW12 --extra-experiments-json outputs/ood_followup/independent_stf3_experiments.json --thresholds-csv outputs/ood_followup/independent_stf3_runs/val_calibrated_thresholds.csv --objective precision_0.95_recall_max --out-dir outputs/ood_followup/independent_stf3_runs
[log] logs/ood_followup/independent_stf3_generator_analysis.log


```text
independent STF3 generator analysis done, elapsed=0.0 min
completed
```

[write] outputs\ood_followup\independent_stf3_runs\generator_error_analysis_precision_0.95_recall_max.csv
[write] outputs\ood_followup\independent_stf3_runs\r5_r7_fn_comparison_precision_0.95_recall_max.csv
[write] outputs\ood_followup\independent_stf3_runs\generator_error_analysis_precision_0.95_recall_max.md
[done] independent STF3 generator analysis, elapsed=0.0 min


,id,method,objective,threshold,val_acc,val_f1,val_precision,val_recall,val_balanced_acc,val_fn,...,test_recall,test_balanced_acc,test_fn,test_fp,default_test_acc,default_test_f1,default_test_precision,default_test_recall,default_test_fn,default_test_fp
0,R7_224_FAKEW12,stf3_new_224_fakew12,max_f1,0.250732,0.990119,0.987179,0.979852,0.994618,0.990976,5,...,0.905148,0.943574,199,27,0.927460,0.934108,0.993022,0.881792,248,13
1,R7_224_FAKEW12,stf3_new_224_fakew12,max_balanced_acc,0.250732,0.990119,0.987179,0.979852,0.994618,0.990976,5,...,0.905148,0.943574,199,27,0.927460,0.934108,0.993022,0.881792,248,13
2,R7_224_FAKEW12,stf3_new_224_fakew12,precision_0.95_recall_max,0.054459,0.981474,0.976303,0.955670,0.997847,0.984590,2,...,0.927073,0.946203,153,52,0.927460,0.934108,0.993022,0.881792,248,13
3,R7_224_FAKEW12,stf3_new_224_fakew12,max_acc,0.250732,0.990119,0.987179,0.979852,0.994618,0.990976,5,...,0.905148,0.943574,199,27,0.927460,0.934108,0.993022,0.881792,248,13
4,R7_224_FAKEW12_WAVEREP_BANK,stf3_new_224_fakew12_waverep_bank,max_f1,0.391479,0.988884,0.985460,0.985991,0.984930,0.988132,14,...,0.859867,0.926600,294,10,0.912451,0.919210,0.995003,0.854147,306,9
5,R7_224_FAKEW12_WAVEREP_BANK,stf3_new_224_fakew12_waverep_bank,max_balanced_acc,0.391479,0.988884,0.985460,0.985991,0.984930,0.988132,14,...,0.859867,0.926600,294,10,0.912451,0.919210,0.995003,0.854147,306,9
6,R7_224_FAKEW12_WAVEREP_BANK,stf3_new_224_fakew12_waverep_bank,precision_0.95_recall_max,0.125244,0.983944,0.979244,0.968421,0.990312,0.985156,9,...,0.886559,0.936613,238,20,0.912451,0.919210,0.995003,0.854147,306,9
7,R7_224_FAKEW12_WAVEREP_BANK,stf3_new_224_fakew12_waverep_bank,max_acc,0.391479,0.988884,0.985460,0.985991,0.984930,0.988132,14,...,0.859867,0.926600,294,10,0.912451,0.919210,0.995003,0.854147,306,9
8,R7_224_F16_FAKEW12,stf3_new_224_f16_fakew12,max_f1,0.197144,0.988884,0.985492,0.983906,0.987083,0.988541,12,...,0.870829,0.930748,271,14,0.908560,0.915489,0.992758,0.849380,316,13
9,R7_224_F16_FAKEW12,stf3_new_224_f16_fakew12,max_balanced_acc,0.197144,0.988884,0.985492,0.983906,0.987083,0.988541,12,...,0.870829,0.930748,271,14,0.908560,0.915489,0.992758,0.849380,316,13


rows=12 path=outputs/ood_followup/independent_stf3_runs/val_calibrated_thresholds.csv


,generator,n,real_n,fake_n,acc,precision,recall,specificity,tn,fp,fn,tp,id,method,threshold
0,MorphStudio,700,0,700,0.991429,1.0,0.991429,NaN,0,0,6,694,R7_224_FAKEW12,stf3_new_224_fakew12,0.054459
1,Show_1,700,0,700,0.994286,1.0,0.994286,NaN,0,0,4,696,R7_224_FAKEW12,stf3_new_224_fakew12,0.054459
2,Sora,56,0,56,0.964286,1.0,0.964286,NaN,0,0,2,54,R7_224_FAKEW12,stf3_new_224_fakew12,0.054459
3,WildScrape,642,0,642,0.780374,1.0,0.780374,NaN,0,0,141,501,R7_224_FAKEW12,stf3_new_224_fakew12,0.054459
4,real_MSRVTT,1500,1500,0,0.965333,0.0,NaN,0.965333,1448,52,0,0,R7_224_FAKEW12,stf3_new_224_fakew12,0.054459
5,MorphStudio,700,0,700,0.988571,1.0,0.988571,NaN,0,0,8,692,R7_224_FAKEW12_WAVEREP_BANK,stf3_new_224_fakew12_waverep_bank,0.125244
6,Show_1,700,0,700,0.945714,1.0,0.945714,NaN,0,0,38,662,R7_224_FAKEW12_WAVEREP_BANK,stf3_new_224_fakew12_waverep_bank,0.125244
7,Sora,56,0,56,0.892857,1.0,0.892857,NaN,0,0,6,50,R7_224_FAKEW12_WAVEREP_BANK,stf3_new_224_fakew12_waverep_bank,0.125244
8,WildScrape,642,0,642,0.710280,1.0,0.710280,NaN,0,0,186,456,R7_224_FAKEW12_WAVEREP_BANK,stf3_new_224_fakew12_waverep_bank,0.125244
9,real_MSRVTT,1500,1500,0,0.986667,0.0,NaN,0.986667,1480,20,0,0,R7_224_FAKEW12_WAVEREP_BANK,stf3_new_224_fakew12_waverep_bank,0.125244


rows=15 path=outputs/ood_followup/independent_stf3_runs/generator_error_analysis_precision_0.95_recall_max.csv


,generator,n,real_n,fake_n,acc,precision,recall,specificity,tn,fp,fn,tp,id,method,threshold
0,MorphStudio,700,0,700,0.991429,1.0,0.991429,NaN,0,0,6,694,R7_224_FAKEW12,stf3_new_224_fakew12,0.054459
1,Show_1,700,0,700,0.994286,1.0,0.994286,NaN,0,0,4,696,R7_224_FAKEW12,stf3_new_224_fakew12,0.054459
2,Sora,56,0,56,0.964286,1.0,0.964286,NaN,0,0,2,54,R7_224_FAKEW12,stf3_new_224_fakew12,0.054459
3,WildScrape,642,0,642,0.780374,1.0,0.780374,NaN,0,0,141,501,R7_224_FAKEW12,stf3_new_224_fakew12,0.054459
4,real_MSRVTT,1500,1500,0,0.965333,0.0,NaN,0.965333,1448,52,0,0,R7_224_FAKEW12,stf3_new_224_fakew12,0.054459
5,MorphStudio,700,0,700,0.988571,1.0,0.988571,NaN,0,0,8,692,R7_224_FAKEW12_WAVEREP_BANK,stf3_new_224_fakew12_waverep_bank,0.125244
6,Show_1,700,0,700,0.945714,1.0,0.945714,NaN,0,0,38,662,R7_224_FAKEW12_WAVEREP_BANK,stf3_new_224_fakew12_waverep_bank,0.125244
7,Sora,56,0,56,0.892857,1.0,0.892857,NaN,0,0,6,50,R7_224_FAKEW12_WAVEREP_BANK,stf3_new_224_fakew12_waverep_bank,0.125244
8,WildScrape,642,0,642,0.710280,1.0,0.710280,NaN,0,0,186,456,R7_224_FAKEW12_WAVEREP_BANK,stf3_new_224_fakew12_waverep_bank,0.125244
9,real_MSRVTT,1500,1500,0,0.986667,0.0,NaN,0.986667,1480,20,0,0,R7_224_FAKEW12_WAVEREP_BANK,stf3_new_224_fakew12_waverep_bank,0.125244


## 12. 最终判断顺序

建议按这个顺序决定论文主模型：

1. 先比较 R7 默认阈值、R7 val-calibrated、R5 val-calibrated。
2. 再看 generator-level FN，尤其 WildScrape 和 Sora。
3. 如果 STF3-only ensemble 或 branch calibration 超过 R7 单模型，可以作为增强后处理；否则保持 R7 单模型更简洁。
4. R5_224 / R7_224 和 R5_224_FAKEW12 / R7_224_FAKEW12 用来验证三分支贡献；其中 R5_224_FAKEW12 是最终主模型的公平对照。
5. 不要只凭单个 ACC 或单个 Recall 下结论；同时看 Precision、FN 来源和 val/test 一致性。
